# 01 — Data layer exploration (Step 1: what does each source actually contain?)

Built part by part: each part asks one question of one source, the output is read with the
owner, a note is written, then the next part is added. Runs on Kaggle with internet on.
Versions pinned to `uv.lock` (CLAUDE.md §8).

In [1]:
from pathlib import Path

if Path("/kaggle/working").exists():  # Kaggle only; locally uv.lock provides these
    get_ipython().run_line_magic("pip", "install -q pandas==3.0.5 duckdb==1.5.5")

In [2]:
import subprocess
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

ON_KAGGLE = Path("/kaggle/working").exists()
REPO = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "pyproject.toml").exists())  # kernel may start in notebooks/
WORK = Path("/kaggle/working") if ON_KAGGLE else REPO / "data" / "cache"  # local: reuse the project caches
TM_URL = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb"
TM_DB = WORK / "transfermarkt.duckdb" if ON_KAGGLE else REPO / "data" / "raw" / "transfermarkt.duckdb"
if not TM_DB.exists():  # 211 MB weekly snapshot of dcaribou/transfermarkt-datasets
    subprocess.run(["curl", "-sL", "-o", str(TM_DB), TM_URL], check=True)
con = duckdb.connect(str(TM_DB), read_only=True)
print("snapshot:", con.execute("SELECT * FROM version").fetchall())

snapshot: [('154367dfa6d6eb0b86332e332f9df0a080c7ddce',)]


## Part 1 — Transfermarkt `player_valuations`: how are valuations spaced, are there duplicate dates?

Feeds the `value_at` rule (which valuation counts as "the value at date X").

In [3]:
con.execute("""
SELECT player_id, COUNT(*) AS n, COUNT(DISTINCT date) AS distinct_dates, MIN(date) AS first, MAX(date) AS last
FROM player_valuations GROUP BY 1 ORDER BY n DESC LIMIT 5""").df()

,player_id,n,distinct_dates,first,last
0,39333,57,57,2006-05-21,2025-12-29
1,44162,55,55,2007-09-20,2025-10-21
2,51894,53,53,2007-08-23,2026-05-21
3,35047,53,53,2006-09-24,2026-06-05
4,42460,52,52,2006-08-20,2026-05-28


In [4]:
busiest_player_id = con.execute(
    "SELECT player_id FROM player_valuations GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 1"
).fetchone()[0]
valuations = con.execute(
    f"SELECT * FROM player_valuations WHERE player_id = {busiest_player_id} ORDER BY date"
).df()
valuations["gap_days"] = valuations.date.diff().dt.days
valuations.tail(12)

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id,gap_days
45,39333,2023-03-14,475000,Basaksehir FK,6890,TR1,138.0
46,39333,2023-06-08,375000,Basaksehir FK,6890,TR1,86.0
47,39333,2023-10-20,375000,Eyüpspor,7160,TR1,134.0
48,39333,2023-12-29,325000,Eyüpspor,7160,TR1,70.0
49,39333,2024-03-26,300000,Eyüpspor,7160,TR1,88.0
50,39333,2024-06-13,225000,Eyüpspor,7160,TR1,79.0
51,39333,2024-10-04,175000,Eyüpspor,7160,TR1,113.0
52,39333,2024-12-23,175000,Eyüpspor,7160,TR1,80.0
53,39333,2025-03-20,150000,Eyüpspor,7160,TR1,87.0
54,39333,2025-06-18,125000,Eyüpspor,7160,TR1,90.0


In [5]:
con.execute("""
SELECT quantile_cont(gap, [0.1, 0.5, 0.9]) AS gap_days_p10_p50_p90, COUNT(*) AS gaps
FROM (SELECT date_diff('day', LAG(date) OVER (PARTITION BY player_id ORDER BY date), date) AS gap
      FROM player_valuations) WHERE gap IS NOT NULL""").df()

,gap_days_p10_p50_p90,gaps
0,"[81.0, 168.0, 268.0]",614773


**Note (Part 1, read 2026-08-27).** Kaggle image pins pandas 2.2.2; the 3.0.5 pin installs with
conflict warnings but the notebook runs on 3.0.5 (pandas imported after the install), so parity
with `uv.lock` holds. Snapshot `154367d`. No duplicate valuation dates. Each valuation row carries
the club at that date (`current_club_id`) — a second source of club-at-time. Revaluations follow a
March/June/October/December rhythm: gaps p10 = 81, median = 168, p90 = 268 days (614,773 gaps).
A June update exists for most players, so "last valuation on or before 1 July" is the candidate
rule — but p90 = 268 days means it can be nine months stale for some. Decision deferred to
Part 1b: staleness at 1 July, per Big-5 season.

## Part 1b — How stale is "last valuation on or before 1 July" for Big-5 players?

Candidates for `value_at`: (a) last valuation on or before the date; (b) nearest valuation within
±30 days, else NaN; (c) interpolate between the two surrounding valuations.
Criterion: (a) is acceptable if the median staleness at 1 July is under ~45 days and the share
older than 180 days is small; otherwise (b) or (c).

In [6]:
BIG5 = ("GB1", "ES1", "IT1", "L1", "FR1")
big5_sql = ",".join(f"'{c}'" for c in BIG5)
con.execute(f"""
WITH big5_players AS (
  SELECT DISTINCT a.player_id, CAST(g.season AS INTEGER) AS season
  FROM appearances a JOIN games g ON a.game_id = g.game_id
  WHERE g.competition_id IN ({big5_sql}) AND CAST(g.season AS INTEGER) BETWEEN 2014 AND 2025
),
at_july AS (
  SELECT p.player_id, p.season, MAKE_DATE(p.season, 7, 1) AS july1,
         MAX(v.date) FILTER (WHERE v.date <= MAKE_DATE(p.season, 7, 1)) AS last_before,
         MIN(v.date) FILTER (WHERE v.date >  MAKE_DATE(p.season, 7, 1)) AS first_after
  FROM big5_players p LEFT JOIN player_valuations v ON v.player_id = p.player_id
  GROUP BY 1, 2
)
SELECT season, COUNT(*) AS player_seasons,
       SUM(last_before IS NULL) AS no_valuation_before,
       quantile_cont(date_diff('day', last_before, july1), 0.5) AS stale_days_median,
       quantile_cont(date_diff('day', last_before, july1), 0.9) AS stale_days_p90,
       AVG(CAST(date_diff('day', last_before, july1) > 180 AS INTEGER)) AS share_older_than_180d,
       AVG(CAST(date_diff('day', last_before, july1) <= 30 OR date_diff('day', july1, first_after) <= 30 AS INTEGER)) AS share_within_30d
FROM at_july GROUP BY 1 ORDER BY 1""").df()

,season,player_seasons,no_valuation_before,stale_days_median,stale_days_p90,share_older_than_180d,share_within_30d
0,2014,2476,152.0,159.0,238.0,0.105422,0.544244
1,2015,2639,170.0,0.0,25.0,0.017416,0.951367
2,2016,2613,170.0,137.0,179.0,0.026607,0.818256
3,2017,2566,127.0,23.0,30.0,0.019270,0.951020
4,2018,2519,128.0,27.0,39.0,0.017148,0.583437
5,2019,2609,163.0,25.0,28.0,0.011447,0.967585
6,2020,2685,200.0,84.0,84.0,0.035815,0.152427
7,2021,2502,62.0,23.0,35.0,0.011475,0.836885
8,2022,2721,144.0,24.0,32.0,0.007761,0.765776
9,2023,2711,147.0,15.0,22.0,0.014431,0.979337


**Note (Part 1b, read 2026-08-27).** Staleness of "last valuation on or before 1 July" for Big-5
player-seasons: median 15–28 days and p90 22–39 in nine of twelve seasons (the June revaluation
lands before the cut); share older than 180 days 0.8–3.6%. Three seasons are systematically
staler — 2014-15 (median 159 d, 10.5% > 180 d), 2016-17 (median 137 d), 2020-21 (median 84 d,
the COVID revaluation pause). `share_within_30d` swings 15–98% only because the June update date
moves relative to 1 July. 2–8% of player-seasons (62–213) have no valuation before 1 July at all
— debutants — and get `NaN`.

**DECIDED — `value_at` = last valuation on or before the date**, stored with `value_age_days`
so staleness is visible; 2014-15, 2016-17 and 2020-21 flagged as staler in the writeup.
Rejected: nearest-within-±30-days (blanks 15–85% of players in swing seasons for a cadence
artefact); interpolation (invents a number between two market decisions).

## Part 2 — Transfermarkt `transfers`: what do a fee of 0 and a NULL fee mean, and why does one player-season have several rows?

Feeds the fee rule in `market` (spec §4.I: cost = fee, or market value when the fee is
undisclosed or the move is free) and how loans are treated.
Criterion: the rule must be readable off the rows — if 0 and NULL each map to one situation
(free / loan / undisclosed) consistently, encode that; if they are mixed, the fee is unusable
below a floor and market value is the cost.

In [7]:
con.execute("DESCRIBE transfers").df()

,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,transfer_date,DATE,YES,None,None,None
2,transfer_season,VARCHAR,YES,None,None,None
3,from_club_id,INTEGER,YES,None,None,None
4,to_club_id,INTEGER,YES,None,None,None
5,from_club_name,VARCHAR,YES,None,None,None
6,to_club_name,VARCHAR,YES,None,None,None
7,transfer_fee,"DECIMAL(18,3)",YES,None,None,None
8,market_value_in_eur,"DECIMAL(18,3)",YES,None,None,None
9,player_name,VARCHAR,YES,None,None,None


In [8]:
con.execute("""
SELECT CASE WHEN transfer_fee IS NULL THEN 'NULL' WHEN transfer_fee = 0 THEN '0' ELSE '>0' END AS fee_class,
       COUNT(*) AS rows_, COUNT(DISTINCT player_id) AS players,
       SUM(CAST(from_club_name = 'Without Club' AS INTEGER)) AS from_without_club,
       SUM(CAST(to_club_name = 'Without Club' AS INTEGER)) AS to_without_club,
       SUM(CAST(from_club_name = 'Retired' OR to_club_name = 'Retired' AS INTEGER)) AS retired
FROM transfers WHERE TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
GROUP BY 1 ORDER BY 1""").df()

,fee_class,rows_,players,from_without_club,to_without_club,retired
0,0,85521,19032,72.0,15.0,0.0
1,>0,16105,9644,0.0,0.0,0.0
2,NULL,49893,20217,2336.0,2967.0,130.0


In [9]:
con.execute("""SELECT player_name, transfer_season, transfer_date, from_club_name, to_club_name, transfer_fee, market_value_in_eur
FROM transfers WHERE transfer_fee = 0 AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
USING SAMPLE 15 ROWS (reservoir, 1)""").df()

,player_name,transfer_season,transfer_date,from_club_name,to_club_name,transfer_fee,market_value_in_eur
0,Amir Syafiz,25/26,2026-06-30,Young Lions,Hougang Utd.,0.0,50000.0
1,Ismaël Koné,25/26,2026-02-01,Sassuolo,Marseille,0.0,14000000.0
2,Kiril Popov,24/25,2024-08-03,Kolos Kovalivka,Chornomorets,0.0,150000.0
3,Sito,24/25,2024-07-04,Asteras Aktor,Intercity,0.0,350000.0
4,Killian Phillips,23/24,2023-08-11,Palace U21,Wycombe,0.0,150000.0
5,Richie Laryea,22/23,2023-06-30,Toronto,Nott'm Forest,0.0,2500000.0
6,Jansen Miller,22/23,2023-05-01,Indiana Hoosier,Long Island RR,0.0,NaN
7,Kanu,22/23,2023-01-01,Botafogo,EC Bahia,0.0,3500000.0
8,Iván López,22/23,2022-12-31,Sacachispas,Ferro,0.0,25000.0
9,Thomas Sabitzer,22/23,2022-08-30,LASK,WSG Tirol,0.0,700000.0


In [10]:
con.execute("""SELECT player_name, transfer_season, transfer_date, from_club_name, to_club_name, transfer_fee, market_value_in_eur
FROM transfers WHERE transfer_fee IS NULL AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
USING SAMPLE 15 ROWS (reservoir, 1)""").df()

,player_name,transfer_season,transfer_date,from_club_name,to_club_name,transfer_fee,market_value_in_eur
0,Adrián Cova,24/25,2025-01-07,Without Club,Krumovgrad,NaN,NaN
1,Al Bahlul Bousahmin,23/24,2024-02-01,Asaria,Al-Ahly,NaN,NaN
2,Mamadou Ousmane Diagne,23/24,2024-01-01,Malmö FF U19,Malmö,NaN,NaN
3,Iñigo Arguibide,23/24,2023-07-01,Osasuna U19,Osasuna B,NaN,NaN


In [11]:
con.execute("""
SELECT rows_per_player_season, COUNT(*) AS player_seasons
FROM (SELECT player_id, transfer_season, COUNT(*) AS rows_per_player_season FROM transfers GROUP BY 1, 2)
GROUP BY 1 ORDER BY 1""").df()

,rows_per_player_season,player_seasons
0,1,96124
1,2,24929
2,3,6643
3,4,1769
4,5,336
5,6,50
6,7,22
7,8,2
8,9,2
9,10,1


In [12]:
con.execute("""
WITH multi AS (
  SELECT player_id, transfer_season FROM transfers GROUP BY 1, 2 HAVING COUNT(*) >= 3
  ORDER BY player_id LIMIT 3)
SELECT t.player_name, t.transfer_season, t.transfer_date, t.from_club_name, t.to_club_name, t.transfer_fee
FROM transfers t JOIN multi USING (player_id, transfer_season)
ORDER BY t.player_name, t.transfer_date""").df()

,player_name,transfer_season,transfer_date,from_club_name,to_club_name,transfer_fee
0,Luiz Gustavo,05/06,2005-08-31,Ipanema-AL,CRB U20,0.0
1,Luiz Gustavo,05/06,2006-02-06,CRB U20,Ipanema-AL,0.0
2,Luiz Gustavo,05/06,2006-05-31,Ipanema-AL,CRB U20,0.0
3,Luiz Gustavo,06/07,2006-07-04,CRB U20,Coruripe,0.0
4,Luiz Gustavo,06/07,2006-09-04,Coruripe,CRB U20,0.0
5,Luiz Gustavo,06/07,2006-09-05,CRB U20,Miguelense FC,NaN
6,Luiz Gustavo,06/07,2006-11-28,Miguelense FC,Corinthians-AL,0.0
7,Luiz Gustavo,06/07,2007-04-30,Corinthians-AL,Miguelense FC,0.0
8,Luiz Gustavo,06/07,2007-05-07,Miguelense FC,Corinthians-AL,NaN
9,Luiz Gustavo,06/07,2007-05-08,Corinthians-AL,CRB,0.0


In [13]:
# A known paid transfer and a known loan, to read the encoding directly
con.execute("""SELECT player_name, transfer_season, transfer_date, from_club_name, to_club_name, transfer_fee, market_value_in_eur
FROM transfers WHERE player_name IN ('Declan Rice', 'João Félix', 'Kylian Mbappé') ORDER BY player_name, transfer_date""").df()

,player_name,transfer_season,transfer_date,from_club_name,to_club_name,transfer_fee,market_value_in_eur
0,Declan Rice,13/14,2013-07-01,Chelsea Youth,West Ham Yth.,0.0,NaN
1,Declan Rice,15/16,2015-07-01,West Ham Yth.,West Ham U18,NaN,NaN
2,Declan Rice,16/17,2016-07-01,West Ham U18,West Ham U23,NaN,NaN
3,Declan Rice,17/18,2017-07-01,West Ham U23,West Ham,NaN,NaN
4,Declan Rice,23/24,2023-07-15,West Ham,Arsenal,116600000.0,90000000.0
5,João Félix,08/09,2008-07-01,Pestinhas Form.,FC Porto Youth,NaN,NaN
6,João Félix,11/12,2011-07-01,FC Porto Youth,Dragon Force F.,NaN,NaN
7,João Félix,12/13,2012-07-01,Dragon Force F.,DragonForce U15,NaN,NaN
8,João Félix,13/14,2013-07-01,DragonForce U15,FC Porto U15,NaN,NaN
9,João Félix,14/15,2014-07-01,FC Porto U15,Padroense U17,NaN,NaN


**Note (Part 2, read 2026-08-27).** `>0` = disclosed fee (Rice 116.6M, Félix 127.2M, Mbappé 180M).
`0` = loan out, loan return (mirror row dated 30 June) *or* free transfer (Mbappé → Real 2024) —
indistinguishable from the fee alone. `NULL` = youth/reserve promotion, "Without Club", retirement:
not a market transaction. 26% of player-seasons have 2+ rows, overwhelmingly loan + return.
The transfer row carries `market_value_in_eur` at transfer time. Decision deferred to Part 2b:
can loans be separated from free transfers structurally (a later mirror move B→A)?

## Part 2b — Can a 0-fee move be classified as a loan by its mirror move?

Rule under test: a move A→B is a loan if the same player later moves B→A. Criterion: the rule is
usable if fee-0 moves mirror often and fee>0 moves almost never do (a paid transfer that is
"mirrored" would be a re-purchase, which is rare).

In [14]:
con.execute("""
WITH moves AS (
  SELECT player_id, transfer_date, from_club_id, to_club_id, transfer_fee,
         CASE WHEN transfer_fee IS NULL THEN 'NULL' WHEN transfer_fee = 0 THEN '0' ELSE '>0' END AS fee_class
  FROM transfers WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
),
mirrored AS (
  SELECT m.*, MIN(r.transfer_date) AS return_date
  FROM moves m LEFT JOIN moves r
    ON r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
   AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH
  GROUP BY ALL
)
SELECT fee_class, COUNT(*) AS moves,
       AVG(CAST(return_date IS NOT NULL AS INTEGER)) AS share_with_mirror_within_24m,
       quantile_cont(date_diff('day', transfer_date, return_date), 0.5) AS mirror_gap_days_median
FROM mirrored GROUP BY 1 ORDER BY 1""").df()

,fee_class,moves,share_with_mirror_within_24m,mirror_gap_days_median
0,0,85521,0.367886,181.0
1,>0,16105,0.034586,189.0
2,NULL,49893,0.021406,263.5


In [15]:
# The mirror rows themselves: are returns dated on season ends (30 June), and do they carry fee 0?
con.execute("""
WITH moves AS (
  SELECT player_id, transfer_date, from_club_id, to_club_id, transfer_fee FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
)
SELECT strftime(r.transfer_date, '%m-%d') AS return_day, COUNT(*) AS returns,
       AVG(CAST(r.transfer_fee = 0 AS INTEGER)) AS share_fee_0
FROM moves m JOIN moves r
  ON r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
 AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH
WHERE m.transfer_fee = 0
GROUP BY 1 ORDER BY 2 DESC LIMIT 8""").df()

,return_day,returns,share_fee_0
0,06-30,14183,1.000000
1,12-31,3921,1.000000
2,07-01,1954,0.319136
3,05-31,1612,1.000000
4,11-30,770,1.000000
5,01-01,694,0.630088
6,01-31,620,0.960331
7,08-01,350,0.948949


In [16]:
# Fee-0 moves with NO mirror: free transfers? Spot-check against contract logic — 1 July dates dominate?
con.execute("""
WITH moves AS (
  SELECT player_id, player_name, transfer_date, from_club_id, to_club_id, from_club_name, to_club_name,
         transfer_fee, market_value_in_eur FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
),
no_mirror AS (
  SELECT m.* FROM moves m LEFT JOIN moves r
    ON r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
   AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH
  WHERE m.transfer_fee = 0 AND r.player_id IS NULL
)
SELECT player_name, transfer_date, from_club_name, to_club_name, market_value_in_eur
FROM no_mirror WHERE market_value_in_eur >= 5000000 ORDER BY hash(player_id) LIMIT 12  -- deterministic sample after the filter""").df()

,player_name,transfer_date,from_club_name,to_club_name,market_value_in_eur
0,Donny van de Beek,2024-06-30,Frankfurt,Man Utd,5000000.0
1,Donny van de Beek,2022-05-31,Everton,Man Utd,25000000.0
2,Houssem Aouar,2023-07-01,Lyon,Roma,13000000.0
3,Christos Mandas,2026-06-30,Bournemouth,Lazio,8000000.0
4,Henry Onyekuru,2021-06-30,Galatasaray,Monaco,7000000.0
5,Ebenezer Akinsanmiro,2026-06-30,Pisa,Inter,7000000.0
6,Donyell Malen,2026-06-30,Roma,Aston Villa,45000000.0
7,Boubacar Kamara,2022-07-01,Marseille,Aston Villa,25000000.0
8,Hans Nicolussi Caviglia,2026-06-30,Parma,Venezia,6000000.0
9,Hans Nicolussi Caviglia,2026-01-28,Fiorentina,Venezia,7000000.0


**Note (Part 2b, read 2026-08-27).** Mirror rule: 36.8% of fee-0 moves have a B→A mirror within
24 months (median gap 181 d); fee>0 3.5% (paid loans); NULL 2.1%. Returns cluster on window ends —
30 Jun (14,183, 100% fee 0), 31 Dec (3,921), 31 May, 30 Nov, 31 Jan; only 1 Jul / 1 Jan are mixed.
DuckDB's trailing `USING SAMPLE` samples *before* the WHERE (two earlier sample cells came back
short/empty); replaced by `ORDER BY hash(player_id) LIMIT n`. Two cases still to classify: the
return rows themselves, and loan-with-option (0-fee A→B followed by a paid A→B).

## Part 2c — Full structural classification of transfer rows

Candidate rule, fee-independent: `loan_return` if the row mirrors a move within the previous 24
months; else `loan_out` if a B→A mirror follows within 24 months; else `loan_then_bought` if a paid
A→B follows within 24 months (option exercised); else `free` when fee = 0; `paid` when fee > 0;
`internal` when fee is NULL. Criterion: the classes must be exhaustive, the free class must look
like genuine free transfers on inspection, and `loan_then_bought` must be non-trivial.

In [17]:
con.execute("""
WITH moves AS (
  SELECT player_id, player_name, transfer_date, from_club_id, to_club_id, from_club_name, to_club_name,
         transfer_fee, market_value_in_eur FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
),
flags AS (
  SELECT m.*,
    EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id
            AND r.to_club_id = m.from_club_id AND r.transfer_date < m.transfer_date
            AND r.transfer_date >= m.transfer_date - INTERVAL 24 MONTH) AS is_return,
    EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id
            AND r.to_club_id = m.from_club_id AND r.transfer_date > m.transfer_date
            AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH) AS has_mirror_after,
    EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.from_club_id
            AND r.to_club_id = m.to_club_id AND r.transfer_fee > 0 AND r.transfer_date > m.transfer_date
            AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH) AS bought_after
  FROM moves m
),
classed AS (
  SELECT *, CASE
    WHEN is_return THEN 'loan_return'
    WHEN has_mirror_after THEN 'loan_out'
    WHEN transfer_fee = 0 AND bought_after THEN 'loan_then_bought'
    WHEN transfer_fee = 0 THEN 'free'
    WHEN transfer_fee > 0 THEN 'paid'
    ELSE 'internal' END AS kind
  FROM flags
)
SELECT kind, COUNT(*) AS rows_, COUNT(DISTINCT player_id) AS players,
       quantile_cont(market_value_in_eur, 0.5) AS value_median
FROM classed GROUP BY 1 ORDER BY 2 DESC""").df()

,kind,rows_,players,value_median
0,internal,47119,19951,250000.0
1,loan_return,33087,13365,500000.0
2,free,31024,14052,300000.0
3,loan_out,26859,13365,500000.0
4,paid,13430,8806,1500000.0


In [18]:
# Inspect the 'free' class for players worth >= 5m: do they read as contract expiries?
con.execute("""
WITH moves AS (
  SELECT player_id, player_name, transfer_date, from_club_id, to_club_id, from_club_name, to_club_name,
         transfer_fee, market_value_in_eur FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
),
free AS (
  SELECT m.* FROM moves m
  WHERE m.transfer_fee = 0
    AND NOT EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id
                    AND r.to_club_id = m.from_club_id AND abs(date_diff('day', r.transfer_date, m.transfer_date)) <= 730)
    AND NOT EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.from_club_id
                    AND r.to_club_id = m.to_club_id AND r.transfer_fee > 0 AND r.transfer_date > m.transfer_date
                    AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH)
)
SELECT player_name, transfer_date, from_club_name, to_club_name, market_value_in_eur
FROM free WHERE market_value_in_eur >= 5000000 ORDER BY hash(player_id) LIMIT 15""").df()

,player_name,transfer_date,from_club_name,to_club_name,market_value_in_eur
0,Houssem Aouar,2023-07-01,Lyon,Roma,13000000.0
1,Boubacar Kamara,2022-07-01,Marseille,Aston Villa,25000000.0
2,Alexander Nübel,2020-07-01,FC Schalke 04,Bayern Munich,9500000.0
3,Alexander Nübel,2023-07-25,Bayern Munich,Stuttgart,8000000.0
4,Alexander Nübel,2026-06-30,Stuttgart,Bayern Munich,12000000.0
5,Kevin Diks,2025-07-01,Copenhagen,Mönchengladbach,5000000.0
6,Nikola Maksimović,2021-08-31,SSC Napoli,Genoa,10000000.0
7,Christopher Lenz,2021-07-01,Union Berlin,Frankfurt,5000000.0
8,Tanguy Nianzou,2020-07-01,PSG,Bayern Munich,11000000.0
9,João Mário,2021-07-13,Inter,Benfica,12000000.0


In [19]:
# And the 'loan_then_bought' class: does it look like exercised options?
con.execute("""
WITH moves AS (
  SELECT player_id, player_name, transfer_date, from_club_id, to_club_id, from_club_name, to_club_name,
         transfer_fee, market_value_in_eur FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
)
SELECT m.player_name, m.transfer_date AS loan_date, r.transfer_date AS purchase_date, m.from_club_name, m.to_club_name,
       r.transfer_fee AS purchase_fee, m.market_value_in_eur
FROM moves m JOIN moves r ON r.player_id = m.player_id AND r.from_club_id = m.from_club_id AND r.to_club_id = m.to_club_id
  AND r.transfer_fee > 0 AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH
WHERE m.transfer_fee = 0 ORDER BY hash(m.player_id) LIMIT 10""").df()

,player_name,loan_date,purchase_date,from_club_name,to_club_name,purchase_fee,market_value_in_eur
0,Leandro Lozano,2022-02-02,2023-01-02,Boston River,Nacional,598000.0,400000.0
1,Iván Angulo,2022-07-25,2024-01-31,Palmeiras,Orlando,1400000.0,800000.0
2,Louis Patris,2024-09-04,2025-07-01,RSC Anderlecht,Sint-Truiden,1750000.0,2000000.0
3,Cenk Özkacar,2022-08-23,2023-07-01,Lyon,Valencia,5000000.0,2000000.0
4,Daniel Luna,2023-01-31,2023-07-01,Deportivo Cali,RCD Mallorca B,1000000.0,700000.0
5,Pep Biel,2024-08-14,2026-01-26,Olympiacos,Charlotte,3500000.0,3000000.0
6,Dylan Levitt,2021-08-20,2022-07-07,Man Utd U23,Dundee United,355000.0,650000.0
7,George Edmundson,2024-08-30,2025-01-25,Ipswich,Middlesbrough,710000.0,350000.0
8,Ali Sowe,2021-02-16,2021-07-01,CSKA-Sofia,Rostov,3000000.0,2000000.0
9,Sebastiano Luperto,2021-08-13,2023-07-01,Napoli,FC Empoli,2500000.0,2200000.0


**Note (Part 2c, read 2026-08-27).** Fee-independent mirror rule, 2014-15 → 2025-26:
`internal` 47,119 · `loan_return` 33,087 · `free` 31,024 · `loan_out` 26,859 · `paid` 13,430.
`loan_then_bought` is empty — Transfermarkt records an exercised option as loan → return → paid,
so the mirror rule covers it; class dropped. The `free` class reads as contract expiries in 13/15
(Aouar, Kamara, Donnarumma…); the two misses are one three-year loan (Nübel, Bayern → Stuttgart
2023-25-26) whose return falls outside the 24-month window. Open: the mirror window, and whether
`internal` hides club-to-club moves with an undisclosed (NULL) fee.

## Part 2d — Mirror window and undisclosed fees

Candidates for the window: 24, 36, 48 months. Criterion: choose the smallest window that captures
nearly all loan lengths (share of mirrors beyond it < ~2%) while the paid-side mirror share (false
loans) stays near its 24-month level. Undisclosed fees: if NULL-fee rows between two senior clubs
are material, they get their own class `undisclosed` and are priced at market value (spec §4.I).

In [20]:
con.execute("""
WITH moves AS (
  SELECT player_id, transfer_date, from_club_id, to_club_id, transfer_fee FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
),
gaps AS (
  SELECT m.transfer_fee, MIN(date_diff('day', m.transfer_date, r.transfer_date)) AS gap_days
  FROM moves m JOIN moves r ON r.player_id = m.player_id AND r.from_club_id = m.to_club_id
    AND r.to_club_id = m.from_club_id AND r.transfer_date > m.transfer_date
    AND r.transfer_date <= m.transfer_date + INTERVAL 48 MONTH
  GROUP BY m.player_id, m.transfer_date, m.from_club_id, m.to_club_id, m.transfer_fee
)
SELECT CASE WHEN transfer_fee = 0 THEN '0' WHEN transfer_fee > 0 THEN '>0' ELSE 'NULL' END AS fee_class,
       COUNT(*) AS mirrored_moves_48m,
       AVG(CAST(gap_days > 730 AS INTEGER)) AS share_beyond_24m,
       AVG(CAST(gap_days > 1095 AS INTEGER)) AS share_beyond_36m,
       quantile_cont(gap_days, [0.5, 0.9, 0.99]) AS gap_p50_p90_p99
FROM gaps GROUP BY 1 ORDER BY 1""").df()

,fee_class,mirrored_moves_48m,share_beyond_24m,share_beyond_36m,gap_p50_p90_p99
0,0,31798,0.010976,0.002201,"[183.0, 364.0, 734.0]"
1,>0,737,0.251018,0.088195,"[365.0, 1063.8, 1399.4799999999996]"
2,NULL,1191,0.130982,0.041142,"[305.0, 740.0, 1283.4999999999995]"


In [21]:
# Paid moves that mirror: how does the false-loan share grow with the window?
con.execute("""
WITH moves AS (
  SELECT player_id, transfer_date, from_club_id, to_club_id, transfer_fee FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
)
SELECT COUNT(*) AS paid_moves,
       AVG(CAST(EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
                        AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH) AS INTEGER)) AS mirror_24m,
       AVG(CAST(EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
                        AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 36 MONTH) AS INTEGER)) AS mirror_36m,
       AVG(CAST(EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
                        AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 48 MONTH) AS INTEGER)) AS mirror_48m
FROM moves m WHERE m.transfer_fee > 0""").df()

,paid_moves,mirror_24m,mirror_36m,mirror_48m
0,16105,0.034586,0.042099,0.045762


In [22]:
# NULL-fee rows between two senior clubs (neither side youth/reserve/without club): undisclosed fees?
con.execute("""
WITH senior AS (
  SELECT * FROM transfers
  WHERE transfer_fee IS NULL AND from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
    AND NOT regexp_matches(from_club_name, '(U1[5-9]|U2[0-3]|Youth|Yth|Jgd|Juvenil|Academy|\\bB$|\\bII$|Without Club|Retired|Unknown)')
    AND NOT regexp_matches(to_club_name,   '(U1[5-9]|U2[0-3]|Youth|Yth|Jgd|Juvenil|Academy|\\bB$|\\bII$|Without Club|Retired|Unknown)')
)
SELECT COUNT(*) AS null_fee_senior_rows, COUNT(DISTINCT player_id) AS players,
       quantile_cont(market_value_in_eur, [0.5, 0.9]) AS value_p50_p90,
       AVG(CAST(market_value_in_eur >= 1000000 AS INTEGER)) AS share_value_ge_1m
FROM senior""").df()

,null_fee_senior_rows,players,value_p50_p90,share_value_ge_1m
0,10887,7142,"[300000.0, 1000000.0]",0.117404


In [23]:
con.execute("""
SELECT player_name, transfer_date, from_club_name, to_club_name, market_value_in_eur FROM transfers
WHERE transfer_fee IS NULL AND from_club_id IS NOT NULL AND to_club_id IS NOT NULL
  AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25 AND market_value_in_eur >= 3000000
  AND NOT regexp_matches(from_club_name, '(U1[5-9]|U2[0-3]|Youth|Yth|Jgd|Juvenil|Academy|\\bB$|\\bII$|Without Club|Retired|Unknown)')
  AND NOT regexp_matches(to_club_name,   '(U1[5-9]|U2[0-3]|Youth|Yth|Jgd|Juvenil|Academy|\\bB$|\\bII$|Without Club|Retired|Unknown)')
ORDER BY hash(player_id) LIMIT 12""").df()

,player_name,transfer_date,from_club_name,to_club_name,market_value_in_eur
0,Franco Petroli,2026-01-15,Godoy Cruz,Lanús,3000000.0
1,Jesús Angulo,2021-07-01,Santos Laguna,Atlas,3500000.0
2,Luis Abram,2023-02-02,Granada CF,Atlanta,4000000.0
3,Fabio Miretti,2022-07-01,Juve Next Gen,Juventus,5000000.0
4,Álvaro Medrán,2024-01-26,Al-Taawoun,Al-Ettifaq,4500000.0
5,Lucas Rodríguez,2024-07-03,Club Tijuana,Querétaro FC,3000000.0
6,Antonio Blanco,2022-07-01,RM Castilla,Real Madrid,5000000.0
7,Junya Ito,2020-07-01,Kashiwa Reysol,Genk,4500000.0
8,Francesco Camarda,2025-06-01,Milan Futuro,AC Milan,10000000.0
9,Dro Fernández,2025-10-01,Barça Atlètic,Barcelona,5000000.0


**Note (Part 2d, read 2026-08-27).** Loan lengths (31,798 mirrored fee-0 moves): median 183 d,
p90 364, p99 734; 1.1% beyond 24 months, 0.22% beyond 36. Paid-side mirrors 3.5% → 4.2% at 36 m,
median gap 365 d — paid loans, correctly loans. NULL fees between "senior" clubs: 10,887 rows,
median value €300k, 11.7% ≥ €1m; sample = reserve-team promotions the name pattern missed (Juve
Next Gen, Castilla, Milan Futuro, Barça Atlètic) and undisclosed non-European moves.

**DECIDED — transfer classification**, fee-independent, **36-month mirror window**:
`loan_return` (mirrors an earlier move) → `loan_out` (a mirror follows) → `paid` (fee > 0) →
`free` (fee = 0) → `undisclosed` (fee NULL, both clubs in `clubs`) → `internal` (fee NULL
otherwise). Cost (spec §4.I): fee for `paid`; market value at transfer for `free` and
`undisclosed`; loans and internal moves are neither purchases nor sales. Rejected: 24-month window
(misses 1.1% of loans incl. multi-year); 48 months (0.2% more loans, more re-purchases swept in);
fee-based loan detection (paid loans exist); name-regex reserve detection (leaky).

## Part 3 — `players.contract_expiration_date` and `game_lineups`

3a: is contract end current-only, and how many players lack it? (feeds whether contract remaining
can be a feature at all — spec §4.D already excludes it from the backtested market model).
3b: which lineup rows count as "in the squad", what positions are recorded, and do lineups agree
with appearances on who played? (feeds the appearances ∪ lineups union in the identity table).

In [24]:
con.execute("""
SELECT last_season, COUNT(*) AS players, SUM(CAST(contract_expiration_date IS NOT NULL AS INTEGER)) AS with_contract_end,
       MIN(contract_expiration_date) AS earliest_end, MAX(contract_expiration_date) AS latest_end
FROM players GROUP BY 1 ORDER BY 1 DESC LIMIT 8""").df()

,last_season,players,with_contract_end,earliest_end,latest_end
0,2025,22292,17221.0,2026-01-10,2035-06-30
1,2024,6075,3908.0,2026-01-31,2031-06-30
2,2023,2092,1455.0,2000-05-31,2029-06-30
3,2022,1877,1186.0,2023-05-31,2028-06-30
4,2021,1817,1357.0,2023-03-15,2027-07-31
5,2020,1843,1182.0,2023-03-31,2027-12-31
6,2019,1566,931.0,2023-01-31,2027-06-30
7,2018,2065,1085.0,2023-01-01,2026-12-31


In [25]:
con.execute("SELECT type, COUNT(*) AS rows_ FROM game_lineups GROUP BY 1").df()

,type,rows_
0,starting_lineup,1780759
1,substitutes,1398257


In [26]:
con.execute("SELECT position, COUNT(*) AS rows_ FROM game_lineups GROUP BY 1 ORDER BY 2 DESC").df()

,position,rows_
0,Centre-Back,551970
1,Centre-Forward,394661
2,Goalkeeper,357844
3,Central Midfield,329361
4,Defensive Midfield,279364
5,Right-Back,230225
6,Left-Back,219387
7,Left Winger,205392
8,Attacking Midfield,203736
9,Right Winger,203171


In [27]:
# Do lineups and appearances agree on who played? All PL 2023-24 games.
con.execute("""
WITH g AS (SELECT game_id FROM games WHERE competition_id = 'GB1' AND season = '2023'),
a AS (SELECT DISTINCT CAST(game_id AS VARCHAR) AS game_id, player_id FROM appearances
      WHERE CAST(game_id AS VARCHAR) IN (SELECT game_id FROM g)),
l AS (SELECT DISTINCT CAST(game_id AS VARCHAR) AS game_id, player_id, type FROM game_lineups
      WHERE CAST(game_id AS VARCHAR) IN (SELECT game_id FROM g))
SELECT
  (SELECT COUNT(*) FROM a) AS appearance_rows,
  (SELECT COUNT(*) FROM l WHERE type = 'starting_lineup') AS starting_rows,
  (SELECT COUNT(*) FROM l WHERE type <> 'starting_lineup') AS bench_rows,
  (SELECT COUNT(*) FROM a LEFT JOIN l USING (game_id, player_id) WHERE l.player_id IS NULL) AS played_but_not_in_lineups,
  (SELECT COUNT(*) FROM l LEFT JOIN a USING (game_id, player_id) WHERE l.type = 'starting_lineup' AND a.player_id IS NULL) AS started_but_no_appearance""").df()

,appearance_rows,starting_rows,bench_rows,played_but_not_in_lineups,started_but_no_appearance
0,11384,8360,6820,0,0


In [28]:
# Minutes for bench players who came on vs. who did not: is 'substitutes' usable as 'in squad' only?
con.execute("""
WITH g AS (SELECT game_id FROM games WHERE competition_id = 'GB1' AND season = '2023')
SELECT l.type, COUNT(*) AS rows_, SUM(CAST(a.player_id IS NOT NULL AS INTEGER)) AS with_appearance,
       quantile_cont(a.minutes_played, 0.5) AS minutes_median
FROM game_lineups l
LEFT JOIN appearances a ON CAST(a.game_id AS VARCHAR) = CAST(l.game_id AS VARCHAR) AND a.player_id = l.player_id
WHERE CAST(l.game_id AS VARCHAR) IN (SELECT game_id FROM g) GROUP BY 1""").df()

,type,rows_,with_appearance,minutes_median
0,substitutes,6820,3024.0,16.0
1,starting_lineup,8360,8360.0,90.0


**Note (Part 3, read 2026-08-27).** 3a: `contract_expiration_date` is current-only — present for
77% of active players (17,221 / 22,292 with `last_season` 2025), stale or absent for anyone whose
last season is older (a 2023 player shows 2000-05-31). Usable only as a descriptive field for
current candidates; never a backtest feature (spec §4.D stands).
3b: lineups and appearances agree exactly on PL 2023-24 — 11,384 appearance rows = 8,360 starters
+ 3,024 used substitutes; 0 played-without-lineup, 0 started-without-appearance; 3,796 unused
substitutes exist only in lineups. `type` ∈ {starting_lineup, substitutes}. `position` is
Transfermarkt's detailed per-match role (Centre-Back … Second Striker; 0.4% generic leftovers).

**DECIDED** — identity's Transfermarkt side = appearances ∪ lineups (both types): adds bench-only
squad members, which is right for matching; minutes always from Understat. Transfermarkt lineup
`position` is a candidate role vocabulary for Step 5a alongside Understat's codes.

## Part 4 — Understat: one match, then the vocabularies

One match's player rows and shots printed whole; then, over the first 30 matches of PL 2023-24,
the result / situation / body-part / position vocabularies and how own goals are recorded;
then the season-level player table and the team-match table (style-vector candidates).

In [29]:
from pathlib import Path

if Path("/kaggle/working").exists():  # Kaggle only; locally uv.lock provides these
    get_ipython().run_line_magic("pip", "install -q soccerdata==1.9.1")

In [30]:
import soccerdata as sd

understat = sd.Understat(leagues=["ENG-Premier League"], seasons=["2023-2024"], data_dir=WORK / "understat")
schedule = understat.read_schedule().reset_index()
print(len(schedule), "matches;", list(schedule.columns))
schedule.head(3)

[08/27/26 21:32:51] INFO     No custom team name replacements found. You can configure these in       ]8;id=6373847;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=6373848;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_config.py#91\91]8;;\
                             /Users/mihailandreev/soccerdata/config/teamname_replacements.json.                    

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=6373854;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=6373855;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_config.py#189\189]8;;\
                             /Users/mihailandreev/soccerdata/config/league_dict.json.                              

                    INFO     Saving cached data to                                                   ]8;id=6373862;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=6373863;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/mihailandreev/football-player-scouting/data/cache/understat                    

[2026-08-27 21:32:51] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: /Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib


                    INFO     Successfully loaded TLS library:                                      ]8;id=6373870;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py\libraries.py]8;;\:]8;id=6373871;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py#397\397]8;;\
                             /Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/si                 
                             te-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib                     

380

matches;

['league', 'season', 'game', 'league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'is_result', 'has_data', 'url']

,league,season,game,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,home_goals,away_goals,home_xg,away_xg,is_result,has_data,url
0,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,1,2023,22275,2023-08-11 19:00:00,92,88,Burnley,Manchester City,MCI,BUR,0,3,0.311032,2.40074,True,True,https://understat.com/match/22275
1,ENG-Premier League,2324,2023-08-12 Arsenal-Nottingham Forest,1,2023,22276,2023-08-12 11:30:00,83,249,Arsenal,Nottingham Forest,NOT,ARS,2,1,0.84262,0.966305,True,True,https://understat.com/match/22276
2,ENG-Premier League,2324,2023-08-12 Bournemouth-West Ham,1,2023,22277,2023-08-12 14:00:00,73,81,Bournemouth,West Ham,WHU,BOU,1,1,1.51025,1.4834,True,True,https://understat.com/match/22277


In [31]:
match_id = schedule.game_id.iloc[0]
player_rows = understat.read_player_match_stats(match_id=[match_id]).reset_index()
print(list(player_rows.columns))
player_rows

['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'team_id', 'player_id', 'position', 'position_id', 'minutes', 'goals', 'own_goals', 'shots', 'xg', 'xg_chain', 'xg_buildup', 'assists', 'xa', 'key_passes', 'yellow_cards', 'red_cards']

,league,season,game,team,player,league_id,season_id,game_id,team_id,player_id,position,position_id,minutes,goals,own_goals,shots,xg,xg_chain,xg_buildup,assists,xa,key_passes,yellow_cards,red_cards
0,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Ameen Al Dakhil,1,2023,22275,92,11699,DC,3,90,0,0,0,0.0,0.122039,0.122039,0,0.0,0,0,0
1,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Anass Zaroury,1,2023,22275,92,11703,Sub,17,24,0,0,1,0.063984,0.063984,0.0,0,0.013008,1,0,1
2,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Benson Manuel,1,2023,22275,92,11702,Sub,17,11,0,0,0,0.0,0.063984,0.063984,0,0.0,0,0,0
3,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Connor Roberts,1,2023,22275,92,5568,DR,2,90,0,0,0,0.0,0.122039,0.122039,0,0.0,0,0,0
4,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Dara O'Shea,1,2023,22275,92,8756,DC,3,90,0,0,0,0.0,0.122039,0.122039,0,0.0,0,0,0
5,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Jacob Bruun Larsen,1,2023,22275,92,5355,Sub,17,24,0,0,1,0.013008,0.063984,0.063984,0,0.0,0,0,0
6,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,James Trafford,1,2023,22275,92,9077,GK,1,90,0,0,0,0.0,0.063984,0.063984,0,0.0,0,0,0
7,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Josh Brownhill,1,2023,22275,92,8323,Sub,17,1,0,0,0,0.0,0.0,0.0,0,0.0,0,0,0
8,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Josh Cullen,1,2023,22275,92,1018,MC,9,90,0,0,0,0.0,0.063984,0.063984,0,0.0,0,0,0
9,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Louis Beyer,1,2023,22275,92,6557,DC,3,79,0,0,0,0.0,0.0,0.0,0,0.0,0,0,0


In [32]:
match_shots = understat.read_shot_events(match_id=[match_id]).reset_index()
print(list(match_shots.columns))
match_shots

['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'date', 'shot_id', 'team_id', 'player_id', 'assist_player_id', 'assist_player', 'xg', 'location_x', 'location_y', 'minute', 'body_part', 'situation', 'result']

,league,season,game,team,player,league_id,season_id,game_id,date,shot_id,team_id,player_id,assist_player_id,assist_player,xg,location_x,location_y,minute,body_part,situation,result
0,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Anass Zaroury,1,2023,22275,2023-08-11 19:00:00,531947,92,11703,603612,Lyle Foster,0.063984,0.817,0.536,79,Left Foot,Open Play,Blocked Shot
1,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Jacob Bruun Larsen,1,2023,22275,2023-08-11 19:00:00,531948,92,5355,603615,Anass Zaroury,0.013008,0.958,0.376,81,<NA>,From Corner,Missed Shot
2,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Luca Koleosho,1,2023,22275,2023-08-11 19:00:00,531932,92,10620,603608,Vitinho,0.058055,0.86,0.635,14,Right Foot,Open Play,Missed Shot
3,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Lyle Foster,1,2023,22275,2023-08-11 19:00:00,531934,92,7498,<NA>,<NA>,0.017112,0.87,0.751,28,Left Foot,Open Play,Missed Shot
4,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Lyle Foster,1,2023,22275,2023-08-11 19:00:00,531936,92,7498,603609,Luca Koleosho,0.097616,0.86,0.455,36,Right Foot,Open Play,Blocked Shot
5,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Zeki Amdouni,1,2023,22275,2023-08-11 19:00:00,531933,92,11701,603609,Luca Koleosho,0.061258,0.886,0.672,17,Left Foot,Open Play,Saved Shot
6,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Manchester City,Aymeric Laporte,1,2023,22275,2023-08-11 19:00:00,531951,88,2498,603627,Julián Álvarez,0.112282,0.913,0.527,94,<NA>,Set Piece,Saved Shot
7,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Manchester City,Erling Haaland,1,2023,22275,2023-08-11 19:00:00,531929,88,8260,603625,Rodri,0.505846,0.937,0.506,3,Left Foot,From Corner,Goal
8,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Manchester City,Erling Haaland,1,2023,22275,2023-08-11 19:00:00,531931,88,8260,603624,Kevin De Bruyne,0.130247,0.952,0.421,10,Right Foot,Open Play,Missed Shot
9,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Manchester City,Erling Haaland,1,2023,22275,2023-08-11 19:00:00,531935,88,8260,603627,Julián Álvarez,0.113214,0.891,0.385,35,Left Foot,Open Play,Goal


In [33]:
sample_ids = schedule.game_id.head(30).tolist()
try:
    sample_shots = understat.read_shot_events(match_id=sample_ids).reset_index()
    sample_players = understat.read_player_match_stats(match_id=sample_ids).reset_index()
    print("shot result:", sample_shots.result.value_counts().to_dict())
    print("shot situation:", sample_shots.situation.value_counts().to_dict())
    print("shot body_part:", sample_shots.body_part.value_counts(dropna=False).to_dict())
    print("player-match positions:", sample_players.position.value_counts().to_dict())
    print("own_goals column total:", int(sample_players.own_goals.sum()), "| rows with minutes == 0:", int((sample_players.minutes == 0).sum()))
    display(sample_shots[sample_shots.result == "Own Goal"].head(5))
    display(sample_players[sample_players.own_goals > 0].head(5))
except Exception as exc:  # diagnostics never raise (CLAUDE.md §8)
    print("vocabulary pull failed:", type(exc).__name__, exc)

shot result:

{'Missed Shot': 305, 'Blocked Shot': 250, 'Saved Shot': 209, 'Goal': 88, 'Shot On Post': 20, 'Own Goal': 1}

shot situation:

{'Open Play': 637, 'From Corner': 145, 'Set Piece': 59, 'Direct Freekick': 20}

shot body_part:

...

1

| rows with minutes == 0:

0

,league,season,game,team,player,league_id,season_id,game_id,date,shot_id,team_id,player_id,assist_player_id,assist_player,xg,location_x,location_y,minute,body_part,situation,result
418,ENG-Premier League,2324,2023-08-19 Tottenham-Manchester United,Manchester United,Lisandro Martínez,1,2023,22290,2023-08-19 16:30:00,533350,89,10802,<NA>,<NA>,0.0,0.047,0.487,82,Right Foot,Open Play,Own Goal


,league,season,game,team,player,league_id,season_id,game_id,team_id,player_id,position,position_id,minutes,goals,own_goals,shots,xg,xg_chain,xg_buildup,assists,xa,key_passes,yellow_cards,red_cards
435,ENG-Premier League,2324,2023-08-19 Tottenham-Manchester United,Manchester United,Lisandro Martínez,1,2023,22290,89,10802,DC,3,90,0,1,0,0.0,0.176897,0.156797,0,0.0201,1,0,0


In [34]:
season_players = understat.read_player_season_stats().reset_index()
print(len(season_players), "players;", list(season_players.columns))
print("season-level position strings:", season_players.position.value_counts().head(15).to_dict())
season_players.head(5)

570

players;

['league', 'season', 'team', 'player', 'league_id', 'season_id', 'team_id', 'player_id', 'position', 'matches', 'minutes', 'goals', 'xg', 'np_goals', 'np_xg', 'assists', 'xa', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'xg_chain', 'xg_buildup']

season-level position strings:

{'D S': 124, 'M S': 117, 'S': 82, 'F M S': 65, 'F S': 47, 'D M S': 36, 'D': 33, 'GK': 30, 'M': 11, 'GK S': 8, 'D F M S': 8, 'F M': 4, 'F': 2, 'D M': 2, 'D F S': 1}

,league,season,team,player,league_id,season_id,team_id,player_id,position,matches,minutes,goals,xg,np_goals,np_xg,assists,xa,shots,key_passes,yellow_cards,red_cards,xg_chain,xg_buildup
0,ENG-Premier League,2324,Arsenal,Aaron Ramsdale,1,2023,83,5603,GK,6,540,0,0.0,0,0.0,0,0.0,0,0,0,0,0.783274,0.783274
1,ENG-Premier League,2324,Arsenal,Ben White,1,2023,83,7298,D S,37,3024,4,2.015914,4,2.015914,4,4.758832,13,38,8,0,20.149292,17.29045
2,ENG-Premier League,2324,Arsenal,Bukayo Saka,1,2023,83,7322,F,35,2990,16,16.807056,10,12.240043,8,11.325986,107,89,4,0,30.357597,13.22862
3,ENG-Premier League,2324,Arsenal,Cédric Soares,1,2023,83,847,S,3,44,0,0.0,0,0.0,0,0.0,0,0,0,0,0.196755,0.196755
4,ENG-Premier League,2324,Arsenal,David Raya,1,2023,83,9676,GK,32,2880,0,0.0,0,0.0,0,0.0,0,0,2,0,7.517592,7.517592


In [35]:
team_matches = understat.read_team_match_stats().reset_index()
print(len(team_matches), "team-matches;", list(team_matches.columns))
team_matches.head(4)

380

team-matches;

['league', 'season', 'game', 'league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'away_points', 'away_expected_points', 'away_goals', 'away_xg', 'away_np_xg', 'away_np_xg_difference', 'away_ppda', 'away_deep_completions', 'home_points', 'home_expected_points', 'home_goals', 'home_xg', 'home_np_xg', 'home_np_xg_difference', 'home_ppda', 'home_deep_completions']

,league,season,game,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,away_points,away_expected_points,away_goals,away_xg,away_np_xg,away_np_xg_difference,away_ppda,away_deep_completions,home_points,home_expected_points,home_goals,home_xg,home_np_xg,home_np_xg_difference,home_ppda,home_deep_completions
0,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,1,2023,22275,2023-08-11 19:00:00,92,88,Burnley,Manchester City,MCI,BUR,3,2.7761,3,2.40074,2.40074,2.089708,15.8,9,0,0.1385,0,0.311032,0.311032,-2.089708,33.071429,4
1,ENG-Premier League,2324,2023-08-12 Arsenal-Nottingham Forest,1,2023,22276,2023-08-12 11:30:00,83,249,Arsenal,Nottingham Forest,NOT,ARS,0,1.4883,1,0.966305,0.966305,0.123685,44.1,4,3,1.1754,2,0.84262,0.84262,-0.123685,4.0,9
2,ENG-Premier League,2324,2023-08-12 Bournemouth-West Ham,1,2023,22277,2023-08-12 14:00:00,73,81,Bournemouth,West Ham,WHU,BOU,1,1.2985,1,1.4834,1.4834,-0.02685,9.733333,8,1,1.3846,1,1.51025,1.51025,0.02685,5.111111,9
3,ENG-Premier League,2324,2023-08-12 Brighton-Luton,1,2023,22278,2023-08-12 14:00:00,220,256,Brighton,Luton,LUT,BRI,0,0.1878,1,1.88594,1.12477,-2.48154,23.0625,5,3,2.7246,4,4.36748,3.60631,2.48154,6.875,17


**Note (Part 4, read 2026-08-27).** Understat per-match player rows: `minutes, goals, own_goals,
shots, xg, xg_chain, xg_buildup, assists, xa, key_passes, cards`, Understat `player_id`/`team_id`,
and `position` — a real role code only for starters (16 codes incl. `FWL/FWR`, `DML/DMR`); every
substitute is `Sub` (27% of rows, 242/902 over 30 matches). Shots: `xg, location_x/y` (0–1),
`minute, body_part, situation, result`; `assist_player_id` is not in the player-id space (use the
name). Vocabularies over 873 shots: result {Missed, Blocked, Saved, Goal, Shot On Post, Own Goal};
situation {Open Play, From Corner, Set Piece, Direct Freekick} — **no "Penalty"** although
`xg ≠ np_xg` in Brighton–Luton proves penalties are present; body_part NA in 17.5% (headers?).
Own goals: shot row `result = Own Goal`, `xg = 0`, at the scorer's own end, plus `own_goals = 1`
on the player row. Season table: 570 players, coarse position strings (`D S`, `F M S`),
`np_goals/np_xg/xg_chain/xg_buildup`. Team-match: wide, both sides — `points, expected_points,
goals, xg, np_xg, np_xg_difference, ppda, deep_completions`; no possession.

**DECIDED:** `Own Goal` shot rows are excluded from every shot-based metric (shots, xG faced,
keeper proxy). Sub-minute role assignment is Step 5a's decision, with 27% as the share it must
handle. Deferred to Part 4b: how penalties are labelled; what the NA body part is.

## Part 4b — Penalties and the NA body part

Understat gives every penalty xG = 0.7612. Criterion: if every shot with xg in [0.75, 0.78] shares
one `situation` label, that label is how penalties arrive and `np_xg` can be reproduced from it;
if the NA body-part shots sit close to goal and never in the Direct Freekick situation, NA = header.

In [36]:
penalty_like = sample_shots[sample_shots.xg.between(0.75, 0.78)]
print(len(penalty_like), "shots with xg in [0.75, 0.78]")
print("their situation:", penalty_like.situation.value_counts().to_dict())
print("their body_part:", penalty_like.body_part.value_counts(dropna=False).to_dict())
print("their result:", penalty_like.result.value_counts().to_dict())
penalty_like[["game", "player", "xg", "location_x", "location_y", "minute", "body_part", "situation", "result"]]

12

shots with xg in [0.75, 0.78]

their situation:

{}

their body_part:

{'Left Foot': 7, 'Right Foot': 5}

their result:

{'Goal': 9, 'Saved Shot': 2, 'Shot On Post': 1}

,game,player,xg,location_x,location_y,minute,body_part,situation,result
87,2023-08-12 Brighton-Luton,João Pedro,0.761169,0.885,0.5,70,Right Foot,<NA>,Goal
103,2023-08-12 Brighton-Luton,Carlton Morris,0.761169,0.885,0.5,80,Right Foot,<NA>,Goal
204,2023-08-13 Brentford-Tottenham,Bryan Mbeumo,0.761169,0.885,0.5,26,Left Foot,<NA>,Goal
316,2023-08-19 Fulham-Brentford,Bryan Mbeumo,0.761169,0.885,0.5,65,Left Foot,<NA>,Goal
370,2023-08-19 Liverpool-Bournemouth,Mohamed Salah,0.761169,0.885,0.5,35,Left Foot,<NA>,Saved Shot
474,2023-08-20 Aston Villa-Everton,Douglas Luiz,0.761169,0.885,0.5,23,Right Foot,<NA>,Goal
501,2023-08-20 West Ham-Chelsea,Enzo Fernández,0.761169,0.885,0.5,42,Right Foot,<NA>,Saved Shot
516,2023-08-20 West Ham-Chelsea,Lucas Paquetá,0.761169,0.885,0.5,94,Left Foot,<NA>,Goal
534,2023-08-21 Crystal Palace-Arsenal,Martin Odegaard,0.761132,0.885,0.5,53,Left Foot,<NA>,Goal
584,2023-08-26 Arsenal-Fulham,Bukayo Saka,0.761169,0.885,0.5,69,Left Foot,<NA>,Goal


In [37]:
na_body = sample_shots[sample_shots.body_part.isna()]
print("NA body-part shots:", len(na_body))
print("situation:", na_body.situation.value_counts().to_dict())
print("location_x quantiles (1 = goal line):", na_body.location_x.quantile([0.1, 0.5, 0.9]).round(3).tolist(),
      "| all shots:", sample_shots.location_x.quantile([0.1, 0.5, 0.9]).round(3).tolist())
print("share that are goals:", round(float((na_body.result == 'Goal').mean()), 3), "| all shots:", round(float((sample_shots.result == 'Goal').mean()), 3))

NA body-part shots:

153

situation:

{'From Corner': 71, 'Open Play': 49, 'Set Piece': 33}

location_x quantiles (1 = goal line):

...

[0.756, 0.874, 0.953]

share that are goals:

0.085

| all shots:

0.101

**Note (Part 4b, read 2026-08-27).** Penalties: soccerdata drops Understat's `Penalty` label, so
they arrive as `situation = NA` with the fixed signature `xg = 0.7612`, `location (0.885, 0.5)` —
12 in 30 matches, exactly the 12 NAs missing from the situation counts (861 of 873). NA body part
(153 shots): never Direct Freekick, 68% from corners/set pieces, closer to goal (median
`location_x` 0.92 vs 0.874), converting 8.5% vs 10.1% — headers and "other body part", both
dropped by the same mapping.

**DECIDED:** `is_penalty` = (`situation` NA ∧ `xg` ∈ [0.75, 0.78] ∧ location ≈ (0.885, 0.5));
`np_xg` = `xg` − penalty xG, verified against Understat's own `np_xg` in the port; NA body part
labelled `head_or_other` (not asserted to be headers).

## Part 5 — Sofascore: one league-season statistics page, without and with a `fields` list

Needs the browser-TLS client (`wrapper-tls-requests`, pinned to `uv.lock`); whether it works on
Kaggle is itself a parity finding. Questions: what one page returns by default; which candidate
fields the endpoint accepts; what `None` means; the page cap; whether keepers share the list.
Then the tournament ids for all 13 leagues, verified through the seasons endpoint.

In [38]:
from pathlib import Path

if Path("/kaggle/working").exists():  # Kaggle only; locally uv.lock provides these
    get_ipython().run_line_magic("pip", "install -q wrapper-tls-requests==1.2.5 requests==2.34.2")

In [39]:
import json
import time

import requests
import tls_requests

UA = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-GB,en;q=0.9",
    "Referer": "https://www.sofascore.com/",
    "Origin": "https://www.sofascore.com",
}
SOFASCORE = "https://api.sofascore.com/api/v1"


def sofascore_get(path, params=None):
    """One polite request through the browser-TLS client (plain clients get 403)."""
    time.sleep(3)
    url = f"{SOFASCORE}{path}" + ("?" + requests.compat.urlencode(params) if params else "")
    response = tls_requests.get(url, headers=UA, timeout=30)
    return response.status_code, response.text


status, body = sofascore_get("/unique-tournament/17/seasons")
print(status, body[:200])
pl_seasons = json.loads(body)["seasons"] if status == 200 else []
pd.DataFrame(pl_seasons).head(12)

200

{"seasons":[{"name":"Premier League 26\/27","year":"26\/27","editor":false,"id":96668},{"name":"Premier League 25\/26","year":"25\/26","editor":false,"id":76986},{"name":"Premier League 24\/25","year"

,name,year,editor,id,seasonCoverageInfo
0,Premier League 26/27,26/27,False,96668,NaN
1,Premier League 25/26,25/26,False,76986,NaN
2,Premier League 24/25,24/25,False,61627,NaN
3,Premier League 23/24,23/24,False,52186,NaN
4,Premier League 22/23,22/23,False,41886,{}
5,Premier League 21/22,21/22,False,37036,NaN
6,Premier League 20/21,20/21,False,29415,NaN
7,Premier League 19/20,19/20,False,23776,NaN
8,Premier League 18/19,18/19,False,17359,NaN
9,Premier League 17/18,17/18,False,13380,NaN


In [40]:
season_2324 = next(s["id"] for s in pl_seasons if s["year"] == "23/24")
status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                             params={"limit": 100, "offset": 0, "accumulation": "total"})
print(status)
default_page = json.loads(body)
print("top-level keys:", list(default_page.keys()))
print("page / pages:", default_page.get("page"), default_page.get("pages"), "| results on page:", len(default_page.get("results", [])))
print("keys of one result:", list(default_page["results"][0].keys()))
pd.json_normalize(default_page["results"]).head(3).T

200

top-level keys:

['results', 'page', 'pages']

page / pages:

1

...

| results on page:

100

keys of one result:

['player', 'team']

,0,1,2
player.name,Rodri,Arijanet Murić,Kevin De Bruyne
player.slug,rodri,arijanet-muric,kevin-de-bruyne
player.userCount,179632,2217,336961
player.gender,M,M,M
player.id,827606,888971,70996
player.fieldTranslations.nameTranslation.ar,رودري,أريجانيت موريش,كيفن دي بروين
player.fieldTranslations.nameTranslation.bn,রডরি,আরিজানেট মুরিচ,কেভিন ডি ব্রুইন
player.fieldTranslations.nameTranslation.hi,रोड्री,अरिजानेट मुरिक,केविन डी ब्रूने
player.fieldTranslations.nameTranslation.ru,Родри,Ариянет Мурич,Кевин Де Брёйне
player.fieldTranslations.shortNameTranslation.ar,رودري,أ. موريش,ك. دي بروين


In [41]:
CANDIDATE_FIELDS = ["minutesPlayed","appearances","rating","goals","assists","expectedGoals","expectedAssists","keyPasses",
    "bigChancesCreated","accuratePasses","accuratePassesPercentage","accurateFinalThirdPasses","accurateLongBalls",
    "accurateCrosses","successfulDribbles","tackles","interceptions","clearances","ballRecovery","possessionWonAttThird",
    "possessionLost","groundDuelsWon","groundDuelsWonPercentage","aerialDuelsWon","aerialDuelsWonPercentage","dribbledPast",
    "fouls","wasFouled","errorLeadToShot","errorLeadToGoal","touches","saves","goalsConceded","savedShotsFromInsideTheBox",
    "savedShotsFromOutsideTheBox","highClaims","punches","penaltyFaced","penaltySave","cleanSheet"]
status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                             params={"limit": 100, "offset": 0, "accumulation": "total", "fields": ",".join(CANDIDATE_FIELDS)})
print(status, body[:300] if status != 200 else "")
fields_page = json.loads(body) if status == 200 else {"results": []}
fields_rows = pd.json_normalize(fields_page["results"])
accepted = [f for f in CANDIDATE_FIELDS if f in fields_rows.columns]
print("accepted:", len(accepted), "of", len(CANDIDATE_FIELDS), "| rejected:", [f for f in CANDIDATE_FIELDS if f not in fields_rows.columns])
print("page / pages:", fields_page.get("page"), fields_page.get("pages"))
print("None counts on this page:", fields_rows[accepted].isna().sum()[lambda s: s > 0].to_dict())
fields_rows.head(3).T

200

accepted:

40

of

40

...

1

6

None counts on this page:

{'expectedGoals': 3}

,0,1,2
minutesPlayed,2938,900,1235
appearances,34,10,18
rating,8.01,8.0,7.93
goals,8,0,4
assists,9,0,10
...,...,...,...
team.fieldTranslations.nameTranslation.ru,Манчестер Сити,Бернли,Манчестер Сити
team.fieldTranslations.shortNameTranslation.ar,مانشستر سيتي,NaN,مانشستر سيتي
team.fieldTranslations.shortNameTranslation.bn,ম্যান সিটি,NaN,ম্যান সিটি
team.fieldTranslations.shortNameTranslation.hi,मैन सिटी,NaN,मैन सिटी


In [42]:
position_col = "player.position"
print("positions on page:", fields_rows[position_col].value_counts().to_dict() if position_col in fields_rows else fields_rows.columns.tolist()[:12])
keepers = fields_rows[fields_rows[position_col] == "G"] if position_col in fields_rows else fields_rows.iloc[0:0]
print("keepers on page:", len(keepers))
keepers[["player.name", "minutesPlayed", "tackles", "saves", "goalsConceded"]].head(3) if len(keepers) else None

positions on page:

['minutesPlayed', 'appearances', 'rating', 'goals', 'assists', 'expectedGoals', 'expectedAssists', 'keyPasses', 'bigChancesCreated', 'accuratePasses', 'accuratePassesPercentage', 'accurateFinalThirdPasses']

keepers on page:

0

In [43]:
# Page depth: how many players does the whole league-season list hold, and does 'pages' cover them?
status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                             params={"limit": 100, "offset": 100 * (fields_page.get("pages", 1) - 1), "accumulation": "total", "fields": "minutesPlayed"})
last_page = json.loads(body)
print(status, "last page:", last_page.get("page"), "of", last_page.get("pages"), "| results:", len(last_page.get("results", [])))
print("→ players listed for PL 23/24 ≈", 100 * (last_page.get("pages", 1) - 1) + len(last_page.get("results", [])))

200

last page:

6

of

6

| results:

70

→ players listed for PL 23/24 ≈

570

In [44]:
# Tournament ids for all 13 leagues: verify each by its recent seasons (a wrong id shows another competition or 404).
TOURNAMENT_GUESSES = {"GB1": 17, "ES1": 8, "IT1": 23, "L1": 35, "FR1": 34,
                      "BE1": 38, "NL1": 37, "PO1": 238, "TR1": 52, "C1": 215, "BRA1": 325, "A1": 45, "DK1": 39}
for comp, tournament_id in TOURNAMENT_GUESSES.items():
    status, body = sofascore_get(f"/unique-tournament/{tournament_id}/seasons")
    years = [s["year"] for s in json.loads(body)["seasons"]][:4] if status == 200 else body[:80]
    print(comp, tournament_id, status, years)

GB1

17

200

['26/27', '25/26', '24/25', '23/24']

ES1

...

['26/27', '25/26', '24/25', '23/24']

DK1

39

200

['26/27', '25/26', '24/25', '23/24']

In [45]:
# Names behind the ids, to confirm the guesses are the right competitions.
for comp, tournament_id in TOURNAMENT_GUESSES.items():
    status, body = sofascore_get(f"/unique-tournament/{tournament_id}")
    name = json.loads(body).get("uniqueTournament", {}).get("name") if status == 200 else body[:80]
    print(comp, tournament_id, status, name)

GB1

17

200

Premier League

ES1

...

Austrian Bundesliga

DK1

39

200

Danish Superliga

**Note (Part 5, run locally 2026-08-27).** Sofascore returns 403 from Kaggle even through the
browser-TLS client — an IP-range block — so all Sofascore work runs locally. From here: the seasons
endpoint lists PL back to 15/16 (ids e.g. 23/24 = 52186); Brazil is calendar-year. All 13
tournament ids verified by name. The default statistics page carries only `player` and `team`
objects — no stats and **no position**; with the field list all 40 candidates are accepted
(page 1 of 6, 100 per page, 570 players for PL 23/24 = Understat's 570 exactly). The only `None`s
on page 1: 3 × `expectedGoals`.

## Part 5b — Keepers via the position filter, what  means, season depth per league

Criterion: keepers must be reachable () with their keeper fields populated;
 must have one meaning (stat not tracked for that player) so it can stay ; each
league's earliest season fixes the pull range.

In [46]:
status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                             params={"limit": 100, "offset": 0, "accumulation": "total", "fields": ",".join(CANDIDATE_FIELDS),
                                     "filters": "position.in.G"})
print(status, body[:200] if status != 200 else "")
keeper_page = json.loads(body) if status == 200 else {"results": []}
keeper_rows = pd.json_normalize(keeper_page["results"])
print("keepers:", len(keeper_rows), "| pages:", keeper_page.get("pages"))
print("None counts among keepers:", keeper_rows[accepted].isna().sum()[lambda s: s > 0].to_dict())
keeper_rows[["player.name", "team.name", "minutesPlayed", "saves", "goalsConceded", "highClaims", "tackles", "expectedGoals"]].head(6)

200

keepers:

40

| pages:

1

None counts among keepers:

{'expectedGoals': 16, 'expectedAssists': 1}

,player.name,team.name,minutesPlayed,saves,goalsConceded,highClaims,tackles,expectedGoals
0,Arijanet Murić,Burnley,900,64,16,19,1,NaN
1,Thomas Strakosha,Brentford,135,6,2,2,0,0.0
2,Alphonse Aréola,West Ham United,2699,138,53,20,1,0.0
3,André Onana,Manchester United,3420,149,58,37,2,0.0
4,Jordan Pickford,Everton,3420,121,51,26,1,0.0
5,Martin Dúbravka,Newcastle United,1994,88,42,18,0,NaN


In [47]:
for position in ["D", "M", "F"]:
    status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                                 params={"limit": 100, "offset": 0, "accumulation": "total", "fields": "minutesPlayed", "filters": f"position.in.{position}"})
    page = json.loads(body) if status == 200 else {}
    print(position, status, "pages:", page.get("pages"), "| first page results:", len(page.get("results", [])))

D

200

pages:

2

| first page results:

...

200

pages:

2

| first page results:

100

In [48]:
none_xg = fields_rows[fields_rows.expectedGoals.isna()]
none_xg[["player.name", "team.name", "minutesPlayed", "appearances", "goals", "tackles", "rating"]]

,player.name,team.name,minutesPlayed,appearances,goals,tackles,rating
1,Arijanet Murić,Burnley,900,10,0,1,8.00
56,Martin Dúbravka,Newcastle United,1994,23,0,0,7.20
60,Mark Travers,Bournemouth,360,4,0,0,7.18


In [49]:
earliest = {}
for comp, tournament_id in TOURNAMENT_GUESSES.items():
    status, body = sofascore_get(f"/unique-tournament/{tournament_id}/seasons")
    years = [s["year"] for s in json.loads(body)["seasons"]] if status == 200 else []
    earliest[comp] = (years[-1] if years else None, len(years))
earliest

{'GB1': ('92/93', 35),
 'ES1': ('1969/1970', 58),
 'IT1': ('1965/1966', 62),
 'L1': ('70/71', 57),
 'FR1': ('70/71', 57),
 'BE1': ('80/81', 47),
 'NL1': ('88/89', 39),
 'PO1': ('02/03', 21),
 'TR1': ('80/81', 47),
 'C1': ('08/09', 19),
 'BRA1': ('2001', 25),
 'A1': ('08/09', 19),
 'DK1': ('08/09', 19)}

**Note (Part 5b, run locally 2026-08-27).** Keepers: 40 for PL 23/24 on one page via
`filters=position.in.G`, keeper fields populated (saves, goals conceded, high claims); position
pages D 2 / M 3 / F 2 / G 1, so pulling per position group yields position at ~8 pages per
league-season. `None` = stat not computed for that player (16 keepers lack `expectedGoals`; Onana
has 0.0 because he took a shot) — stays `NaN`; moot for modelling since xG is Understat's.
Season lists reach back decades (PL 92/93, PT 02/03, CH/AT/DK 08/09); listing ≠ stats, Phase 0
fixed full stats at 15/16, and every list covers 2015-16 → 2025-26.

**DECIDED:** request all 40 accepted fields (which become features is Step 5d); pull per position
filter (G, D, M, F); `None` → `NaN`; seasons 2015-16 → 2025-26 (Brazil 2016 → 2026); Sofascore
pulls run locally only. Tournament ids → `config.py`.

## Part 6 — FotMob: one outfield player's and one keeper's full season-stats JSON; league-level endpoint probe

Plain HTTP (no TLS client). Questions: does Kaggle reach it; the full structure of a season's
stats JSON (sections, per-90, percentiles), for an outfield player and a keeper; whether a
league-season endpoint exists that would replace per-player pulls for Belgium/Denmark.

In [50]:
import re

import requests

PLAIN_UA = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-GB,en;q=0.9",
}


def fotmob_get(url, params=None):
    time.sleep(3)
    response = requests.get(url, params=params, headers=PLAIN_UA, timeout=30)
    return response.status_code, response.text


def fotmob_search(name):
    status, body = fotmob_get("https://apigw.fotmob.com/searchapi/suggest", params={"term": name, "lang": "en"})
    ids = re.findall(r'"id"\s*:\s*"?(\d{4,8})', body)
    return status, (int(ids[0]) if ids else None), body[:200]


def fotmob_page_data(player_id):
    status, body = fotmob_get(f"https://www.fotmob.com/players/{player_id}/x")
    match = re.search(r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', body, re.S)
    return status, (json.loads(match.group(1))["props"]["pageProps"]["data"] if match else None)


fotmob_ids = {}
for name in ["Hans Vanaken", "Simon Mignolet"]:
    status, player_id, head = fotmob_search(name)
    fotmob_ids[name] = player_id
    print(name, status, player_id, "|", head[:120])

Hans Vanaken

200

203666

|

{
  "took": 2,
  "total": 1,
  "squadMemberSuggest": [
    {
      "text": "Hans Vanaken",
      "offset": 0,
      "len

Simon Mignolet

200

37868

|

{
  "took": 2,
  "total": 1,
  "squadMemberSuggest": [
    {
      "text": "Simon Mignolet",
      "offset": 0,
      "l

In [51]:
fotmob_pages = {}
for name, player_id in fotmob_ids.items():
    if player_id is None:
        continue
    status, data = fotmob_page_data(player_id)
    fotmob_pages[name] = data
    print(name, status, "| data keys:", list(data.keys()) if data else None)
    if data:
        recent = [s for s in data["statSeasons"] if s["seasonName"] in ("2023/2024", "2024/2025")]
        for s in recent:
            print("  ", s["seasonName"], [(t["name"], t["entryId"], t.get("hasDeepStats")) for t in s["tournaments"]])

Hans Vanaken

200

| data keys:

['id', 'name', 'birthDate', 'contractEnd', 'isCoach', 'isCaptain', 'gender', 'primaryTeam', 'positionDescription', 'injuryInformation', 'internationalDuty', 'playerInformation', 'mainLeague', 'trophies', 'recentMatches', 'careerHistory', 'traits', 'meta', 'coachStats', 'statSeasons', 'firstSeasonStats', 'status', 'marketValues', 'relatedLinksData', 'nextMatch', 'dataProvider', 'ssr']

2024/2025

...

['id', 'name', 'birthDate', 'isCoach', 'isCaptain', 'gender', 'primaryTeam', 'positionDescription', 'injuryInformation', 'internationalDuty', 'playerInformation', 'mainLeague', 'trophies', 'recentMatches', 'careerHistory', 'traits', 'meta', 'coachStats', 'statSeasons', 'firstSeasonStats', 'status', 'marketValues', 'relatedLinksData', 'nextMatch', 'dataProvider', 'ssr']

2024/2025

[('Belgian Pro League', '1-0', True), ('Champions League', '1-1', True)]

2023/2024

[('Belgian Pro League', '2-0', True), ('Super Cup', '2-1', True), ('Conference League', '2-2', True)]

In [52]:
def fotmob_season_json(player_id, entry_id):
    status, body = fotmob_get("https://www.fotmob.com/api/data/playerStats", params={"playerId": player_id, "seasonId": entry_id})
    return status, (json.loads(body) if status == 200 else body[:200])


def flatten_stats(stats_json):
    rows = []

    def walk(obj, section):
        if isinstance(obj, dict):
            if "title" in obj and "statValue" in obj:
                rows.append({"section": section, **{k: obj.get(k) for k in ["title", "statValue", "per90", "percentileRank", "percentileRankPer90", "statFormat"]}})
            for key, value in obj.items():
                walk(value, obj.get("title", section) if key == "items" else section)
        elif isinstance(obj, list):
            for value in obj:
                walk(value, section)

    walk(stats_json, "")
    return pd.DataFrame(rows)


season_stats = {}
for name, data in fotmob_pages.items():
    season = next((s for s in data["statSeasons"] if s["seasonName"] == "2023/2024"), None)
    if season is None:
        print(name, "no 2023/2024 season"); continue
    domestic = season["tournaments"][0]
    status, stats_json = fotmob_season_json(fotmob_ids[name], domestic["entryId"])
    season_stats[name] = stats_json
    print(name, domestic["name"], status, "| top-level keys:", list(stats_json.keys()) if isinstance(stats_json, dict) else stats_json)

Hans Vanaken

Belgian Pro League

200

| top-level keys:

['sectionOrder', 'shotmap', 'statsSection', 'topStatCard', 'keeperShotmap']

Simon Mignolet

Belgian Pro League

200

| top-level keys:

['sectionOrder', 'keeperShotmap', 'statsSection', 'topStatCard', 'shotmap']

In [53]:
for name, stats_json in season_stats.items():
    if not isinstance(stats_json, dict):
        continue
    flat = flatten_stats(stats_json)
    print(name, "—", len(flat), "stats; sections:", flat.section.value_counts().to_dict())
    display(flat)

Hans Vanaken

—

46

stats; sections:

{'Possession': 11, 'Defending': 11, 'Passing': 10, 'Shooting': 6, '': 6, 'Discipline': 2}

...

—

24

stats; sections:

{'Goalkeeping': 11, '': 6, 'Distribution': 5, 'Discipline': 2}

,section,title,statValue,per90,percentileRank,percentileRankPer90,statFormat
0,Goalkeeping,Saves,71,2.282143,57.142857,14.285714,number
1,Goalkeeping,Save percentage,71.0,71.000000,62.857143,62.857143,percent
2,Goalkeeping,Goals conceded,29,0.932143,34.285714,88.571429,number
3,Goalkeeping,Goals prevented,-0.23,-0.007303,40.000000,45.714286,fraction
4,Goalkeeping,Clean sheets,11,0.353571,82.857143,82.857143,number
5,Goalkeeping,Penalty saves,0,0.000000,0.000000,0.000000,number
6,Goalkeeping,Penalties conceded,1,0.032143,48.571429,62.857143,number
7,Goalkeeping,Penalty save %,0.0,0.000000,25.714286,25.714286,percent
8,Goalkeeping,Error led to goal,0,0.000000,100.000000,100.000000,number
9,Goalkeeping,Acted as sweeper,16,0.514286,88.571429,77.142857,number


In [54]:
# Raw shape of one section, to see what else sits beside title/statValue (e.g. per90 basis, percentile group)
outfield_json = season_stats.get("Hans Vanaken")
print(json.dumps(outfield_json, ensure_ascii=False)[:2500] if isinstance(outfield_json, dict) else outfield_json)

{"sectionOrder": ["top-stat-card", "shotmap", "stats-section"], "shotmap": [{"id": 2571877883, "playerName": "Hans Vanaken", "eventType": "AttemptSaved", "shotType": "LeftFoot", "situation": "RegularPlay", "teamId": 8342, "playerId": 203666, "x": 89.1, "y": 23.4387261191, "min": 15, "period": "FirstHalf", "isOwnGoal": false, "isBlocked": true, "isOnTarget": true, "isSavedOffLine": false, "isFromInsideBox": true, "blockedX": 93.7, "blockedY": 26.4009375, "goalCrossedY": 35.22, "goalCrossedZ": 1.219999994, "expectedGoals": 0.0435284972190857, "onGoalShot": {"x": 0.677248677248676, "y": 0.322751321164021, "zoomRatio": 1}, "box": "InsideBox", "homeTeamId": 8342, "awayTeamId": 8203, "homeTeamName": "Club Brugge", "awayTeamName": "KV Mechelen", "homeScore": 1, "awayScore": 1, "matchId": 4206189, "matchDate": "2023-07-30T16:30:00Z", "teamColor": "#0572FF", "teamColorDark": "#006ad5"}, {"id": 2571879487, "playerName": "Hans Vanaken", "eventType": "AttemptSaved", "shotType": "Header", "situatio

In [55]:
# League-level endpoint probe: does FotMob serve a whole league-season of player stats?
for url, params in [
    ("https://www.fotmob.com/api/leagueseasondeepstats", {"id": 40, "season": "2023/2024", "type": "players", "stat": "tackles"}),
    ("https://www.fotmob.com/api/data/leagueseasondeepstats", {"id": 40, "season": "2023/2024", "type": "players", "stat": "tackles"}),
    ("https://www.fotmob.com/api/data/leagues", {"id": 40, "tab": "stats"}),
    ("https://www.fotmob.com/api/data/leagues", {"id": 40}),
]:
    status, body = fotmob_get(url, params=params)
    print(status, url, params, "|", body[:200].replace("\n", " "))

404

https://www.fotmob.com/api/leagueseasondeepstats

{'id': 40, 'season': '2023/2024', 'type': 'players', 'stat': 'tackles'}

|

<!DOCTYPE html><html lang="en" dir="ltr"><head><meta name="apple-itunes-app" content="app-id=488575683"/><link rel="alternate" href="android-app://com.mobilefootie.wc2010/http"/><link rel="apple-touch

...

200

https://www.fotmob.com/api/data/leagues

{'id': 40}

|

{"tabs":["overview","table","fixtures","stats","transfers","seasons"],"allAvailableSeasons":["2026/2027","2025/2026","2024/2025","2023/2024","2022/2023","2021/2022","2020/2021","2019/2020","2018/2019"

**Note (Part 6, read 2026-08-27).** FotMob is reachable from Kaggle (plain HTTP, 200s). Search →
id (Vanaken 203666, Mignolet 37868); the player page's `statSeasons` lists tournaments per season
with `entryId` and `hasDeepStats`; `/api/data/playerStats?playerId&seasonId=<entryId>` returns
`topStatCard`, `statsSection` and a **`shotmap`** — every shot with xG, xGOT, on-target, body
part, situation, coordinates, match id — i.e. shot-level data for feeder leagues Understat does
not cover. Outfield: 46 stats in Shooting / Passing / Possession / Defending / Discipline, each
with `statValue`, `per90`, `percentileRank` (incl. "Goals conceded while on pitch", "xG against
while on pitch"). Keeper: 24 stats — saves, save %, goals conceded, **goals prevented**, clean
sheets, penalties faced/saved, errors led to goal, sweeper actions, high claims, distribution.
Page data also carries `marketValues`, `contractEnd`, `injuryInformation`, `careerHistory`.
`/api/data/leagueseasondeepstats` exists and wants a season *id* (it lists them) — Part 6b.

**DECIDED:** FotMob keeper "goals prevented" is the second keeper metric alongside the Understat
proxy (Phase 0 Q4). Open until 6b: per-player vs per-league pull for Belgium/Denmark.

## Part 6b — The FotMob league-season endpoint: does it list every player per stat?

`/api/data/leagueseasondeepstats` wants a season *id* (listed in its own response) and a stat
name. Criterion: if one request returns all players of a league-season for one stat (not a
top-N), Belgium/Denmark are pulled per stat (~50 stats × 9 seasons × 2 leagues) instead of per
player; the `leagues?tab=stats` response should name the available stats.

In [56]:
status, body = fotmob_get("https://www.fotmob.com/api/data/leagues", params={"id": 40, "tab": "stats"})
league_json = json.loads(body)
print(status, "top-level keys:", list(league_json.keys()))
stats_block = league_json.get("stats", {})
print("stats keys:", list(stats_block.keys()) if isinstance(stats_block, dict) else type(stats_block))
print(json.dumps(stats_block, ensure_ascii=False)[:2500])

200

top-level keys:

['tabs', 'allAvailableSeasons', 'details', 'seostr', 'QAData', 'table', 'transfers', 'overview', 'stats', 'fixtures', 'playoff', 'seasons']

stats keys:

['players', 'teams', 'seasonStatLinks', 'seasonsWithLinks']

{"players": [{"header": "Top scorer", "participant": {"id": 304860, "name": "Michael Frey", "rank": 1, "ccode": "SUI", "teamId": 9988, "teamName": "Royal Antwerp", "value": 4, "stat": {"name": "goals", "value": 4, "format": "number", "fractions": 0}}, "fetchAllUrl": "https://data.fotmob.com/stats/40/season/37803/goals.json", "topThree": [{"id": 304860, "name": "Michael Frey", "rank": 1, "ccode": "SUI", "teamId": 9988, "teamName": "Royal Antwerp", "value": 4, "stat": {"name": "goals", "value": 4, "format": "number", "fractions": 0}, "teamColors": {"darkMode": "#DB2823", "lightMode": "#DB2823", "fontDarkMode": "rgba(255, 255, 255, 1.0)", "fontLightMode": "rgba(255, 255, 255, 1.0)"}}, {"id": 1353364, "name": "Anthony Valencia", "rank": 2, "ccode": "ECU", "teamId": 9988, "teamName": "Royal Antwerp", "value": 3, "stat": {"name": "goals", "value": 3, "format": "number", "fractions": 0}, "teamColors": {"darkMode": "#DB2823", "lightMode": "#DB2823", "fontDarkMode": "rgba(255, 255, 255, 1.0)", 

In [57]:
status, body = fotmob_get("https://www.fotmob.com/api/data/leagueseasondeepstats",
                          params={"id": 40, "season": "2023/2024", "type": "players", "stat": "tackles"})
season_list = json.loads(body).get("seasons", [])
season_ids = {s["name"]: s["id"] for s in season_list}
print("seasons offered:", season_ids)
belgium_2324 = season_ids.get("2023/2024")
for stat in ["tackles", "interceptions", "goals", "rating", "expected_goals", "xg"]:
    status, body = fotmob_get("https://www.fotmob.com/api/data/leagueseasondeepstats",
                              params={"id": 40, "season": belgium_2324, "type": "players", "stat": stat})
    deep = json.loads(body) if status == 200 else {}
    rows = deep.get("statsData", [])
    print(stat, status, "| players returned:", len(rows), "| first row:", json.dumps(rows[0], ensure_ascii=False)[:300] if rows else None)

seasons offered:

{'2026/2027': 37803, '2025/2026': 27152, '2024/2025': 23650, '2023/2024': 20957, '2022/2023': 17862, '2021/2022': 16419, '2020/2021': 15312, '2019/2020': 14160, '2018/2019': 12768}

tackles

200

| players returned:

...

200

| players returned:

0

| first row:

None

In [58]:
# What stat names does the endpoint know? They usually sit in the response's own metadata.
status, body = fotmob_get("https://www.fotmob.com/api/data/leagueseasondeepstats",
                          params={"id": 40, "season": belgium_2324, "type": "players", "stat": "tackles"})
deep = json.loads(body)
print("response keys:", list(deep.keys()))
for key in deep:
    if key not in ("statsData", "seasons"):
        print(key, "→", json.dumps(deep[key], ensure_ascii=False)[:1500])

response keys:

['statsData', 'seasons', 'statsList', 'leagueDetails', 'type', 'teamName', 'currentSeasonId', 'currentStatName']

statsList

→

[{"name": "goals", "localizedTitleId": "goals_title", "title": "Top scorer", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "goal_assist", "localizedTitleId": "goal_assist_title", "title": "Assists", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "_goals_and_goal_assist", "localizedTitleId": "goals_and_assists", "title": "Goals + Assists", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "rating", "localizedTitleId": "rating_title", "title": "FotMob rating", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "mins_played", "localizedTitleId": "minutes_played", "title": "Minutes played", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "goals_per_90", "localizedTitleId": "goals_per_90_title", "title": "Goals per 90", "category": "Attacking", "localizedCategoryId": "attack"}, {"name": "expected_goals", "localizedTitleId": "expected_goals", "title": "Expected goals (xG)", "category":

...

→

"20957"

currentStatName

→

"tackles"

## Part 6c — FotMob league-season stats: full stat list, the static JSON, completeness, position codes

Criterion: if the static per-stat JSON lists every player of the league-season (`mins_played`
count ≈ the league's player count) with a stable row shape, Belgium/Denmark are pulled per stat
(statsList × seasons); the numeric `position` code needs a mapping read off the data.

In [59]:
stats_list = pd.DataFrame(deep["statsList"])
print(len(stats_list), "stats;", "categories:", stats_list.category.value_counts().to_dict())
print(stats_list[["name", "title", "category"]].to_string())

37

stats;

categories:

{'Attacking': 16, 'Defending': 8, 'Top Stat': 5, 'Goalkeeping': 5, 'Discipline': 3}

                                           name                            title     category
0                                         goals                       Top scorer     Top Stat
1                                   goal_assist                          Assists     Top Stat
2                        _goals_and_goal_assist                  Goals + Assists     Top Stat
3                                        rating                    FotMob rating     Top Stat
4                                   mins_played                   Minutes played     Top Stat
5                                  goals_per_90                     Goals per 90    Attacking
6                                expected_goals              Expected goals (xG)    Attacking
7                         expected_goals_per_90       Expected goals (xG) per 90    Attacking
8                        expected_goalsontarget  Expected goals on target (xGOT)    Attacking
9                          ontarget_scoring_att           Sh

In [60]:
static_url = "https://data.fotmob.com/stats/40/season/{season}/{stat}.json"
for stat in ["mins_played", "rating", "total_tackle", "interception", "won_tackle", "expected_goals"]:
    if stat not in set(stats_list.name):
        print(stat, "— not in statsList"); continue
    status, body = fotmob_get(static_url.format(season=belgium_2324, stat=stat))
    payload = json.loads(body) if status == 200 else {}
    rows = payload.get("TopLists", [{}])[0].get("StatList", []) if "TopLists" in payload else payload.get("statsData", payload if isinstance(payload, list) else [])
    print(stat, status, "| top-level:", list(payload.keys())[:6] if isinstance(payload, dict) else type(payload).__name__, "| rows:", len(rows), "| first:", json.dumps(rows[0], ensure_ascii=False)[:250] if rows else None)

mins_played

200

| top-level:

['TopLists', 'LeagueName']

| rows:

...

['TopLists', 'LeagueName']

| rows:

403

| first:

{"ParticipantName": "Kévin Denkey", "ParticiantId": 820477, "TeamId": 9984, "TeamColor": "#00521f", "StatValue": 25.0, "SubStatValue": 27.0, "MinutesPlayed": 3389, "MatchesPlayed": 38, "StatValueCount": 37, "Rank": 1, "ParticipantCountryCode": "TOG",

In [61]:
status, body = fotmob_get("https://www.fotmob.com/api/data/leagueseasondeepstats", params={"id": 40, "season": belgium_2324, "type": "players", "stat": "mins_played"})
minutes_rows = pd.DataFrame(json.loads(body)["statsData"])
minutes_rows["minutes"] = minutes_rows.statValue.map(lambda d: d["value"])
print(len(minutes_rows), "players with minutes | min minutes listed:", minutes_rows.minutes.min(), "| teams:", minutes_rows.teamId.nunique())
print("position codes:", minutes_rows.position.value_counts().to_dict())
minutes_rows.sort_values("minutes", ascending=False).head(5)

490

players with minutes | min minutes listed:

1

| teams:

16

position codes:

{11: 37, 115: 37, 36: 33, 34: 30, 38: 29, 64: 25, 85: 22, 66: 22, 32: 21, 87: 19, 105: 16, 83: 16, 106: 13, 103: 13, 33: 13, 77: 12, 35: 11, 37: 11, 107: 11, 104: 9, 68: 8, 84: 8, 3: 7, 86: 6, 76: 6, 75: 6, 78: 6, 73: 5, 74: 5, 72: 5, 62: 4, 65: 3, 71: 3, 79: 3, 1: 3, 2: 3, 82: 2, 88: 1, 51: 1, 53: 1, 67: 1, 94: 1, 58: 1, 59: 1}

,id,teamId,name,position,substatValue,statValue,rank,type,minutes
0,972200,9987,Maarten Vandevoordt,11,"{'value': 41, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3690, 'format...",1,players,3690
2,810487,9984,Warleson,11,"{'value': 40, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3600, 'format...",2,players,3600
1,1390101,9997,Matte Smets,36,"{'value': 40, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3600, 'format...",2,players,3600
3,445873,8342,Brandon Mechele,36,"{'value': 40, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3583, 'format...",4,players,3583
4,1280741,9997,Mathias Delorge,64,"{'value': 40, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3551, 'format...",5,players,3551


In [62]:
# Position code → role: read it off known players
known = {"Simon Mignolet": "GK", "Hans Vanaken": "CM/AM", "Kévin Denkey": "ST", "Cameron Puertas": "AM"}
minutes_rows[minutes_rows.name.isin(known)][["name", "teamId", "position", "minutes"]].assign(expected=lambda d: d.name.map(known))

,name,teamId,position,minutes,expected
9,Kévin Denkey,9984,106,3389,ST
10,Hans Vanaken,8342,85,3348,CM/AM
13,Cameron Puertas,7978,77,3323,AM
49,Simon Mignolet,8342,11,2800,GK


## Part 6d — Absence semantics, oldest season, and FotMob league ids

Criterion: a per-90 list's minimum `MinutesPlayed` is its floor — players below it are `NaN`,
players in `mins_played` but absent from a totals list are 0. The oldest offered season must
return the same row shape. League ids are verified by the name the API returns.

In [63]:
def static_stat(league_id, season_id, stat):
    status, body = fotmob_get(f"https://data.fotmob.com/stats/{league_id}/season/{season_id}/{stat}.json")
    rows = json.loads(body)["TopLists"][0]["StatList"] if status == 200 else []
    return status, pd.DataFrame(rows)


minutes_list = static_stat(40, belgium_2324, "mins_played")[1]
for stat in ["total_tackle", "rating", "goals", "expected_goals"]:
    status, rows = static_stat(40, belgium_2324, stat)
    floor = rows.MinutesPlayed.min()
    eligible = int((minutes_list.StatValue >= floor).sum())
    print(f"{stat:16s} {status} rows={len(rows):3d} min MinutesPlayed={floor:5.0f} | players in mins_played with >= that: {eligible}")

total_tackle     200 rows=236 min MinutesPlayed=  437 | players in mins_played with >= that: 341

rating           200 rows=224 min MinutesPlayed=  683 | players in mins_played with >= that: 303

goals            200 rows=233 min MinutesPlayed=   45 | players in mins_played with >= that: 445

expected_goals   200 rows=403 min MinutesPlayed=    3 | players in mins_played with >= that: 481

In [64]:
oldest_season = min(season_ids.values())
status, rows = static_stat(40, oldest_season, "mins_played")
print("oldest season", oldest_season, [k for k, v in season_ids.items() if v == oldest_season], status, "| players:", len(rows), "| columns:", list(rows.columns))

oldest season

12768

['2018/2019']

200

| players:

439

| columns:

['ParticipantName', 'ParticiantId', 'TeamId', 'TeamColor', 'StatValue', 'SubStatValue', 'MinutesPlayed', 'MatchesPlayed', 'StatValueCount', 'Rank', 'ParticipantCountryCode', 'TeamName', 'Positions']

In [65]:
FOTMOB_LEAGUE_GUESSES = {"GB1": 47, "ES1": 87, "IT1": 55, "L1": 54, "FR1": 53,
                         "BE1": 40, "NL1": 57, "PO1": 61, "TR1": 71, "C1": 69, "BRA1": 268, "A1": 38, "DK1": 46}
for comp, league_id in FOTMOB_LEAGUE_GUESSES.items():
    status, body = fotmob_get("https://www.fotmob.com/api/data/leagues", params={"id": league_id})
    details = json.loads(body).get("details", {}) if status == 200 else {}
    seasons = json.loads(body).get("allAvailableSeasons", [])[-1:] if status == 200 else []
    print(comp, league_id, status, details.get("name"), details.get("country"), "| oldest season:", seasons)

GB1

47

200

Premier League

ENG

...

200

Superligaen

DEN

| oldest season:

['2010/2011']

**Note (Parts 6b–6d, run locally 2026-08-27).** `/api/data/leagueseasondeepstats?id&season=<id>&type=players&stat=<name>`
returns whole-league lists for the 37 names in its `statsList` (Attacking 16, Defending 8, Top
Stat 5, Goalkeeping 5 incl. `_goals_prevented`, Discipline 3); the same lists are static JSON at
`data.fotmob.com/stats/{league}/season/{season_id}/{stat}.json`, rows with `ParticipantName,
ParticiantId, TeamId, StatValue, SubStatValue, MinutesPlayed, MatchesPlayed, Rank, Positions`.
`mins_played` lists every player (Belgium 23/24: 490, 16 teams, down to 1 minute). Absence
semantics differ: per-90 lists apply an undocumented eligibility rule (`total_tackle` 236 rows
while 341 players clear its 437-minute minimum) → absent = `NaN`; totals lists have tiny floors
(xG 3 min, goals 45) → absent while in `mins_played` = 0. Oldest offered season (2018/19) has the
same shape. All 13 FotMob league ids verified by name. Local run reproduced Kaggle's Part 1
numbers (parity).

**DECIDED:** FotMob pulled per stat via the static JSON (≈37 × seasons × leagues), never per
player; per-90 absence → `NaN`, totals absence → 0 when the player is in `mins_played`; league
ids → `config.FOTMOB_LEAGUES`. Rejected: per-player pulls for Belgium/Denmark (~20k requests for
the same numbers). Position labels (`Positions`) inspected in the next run.

In [66]:
status, rows = static_stat(40, belgium_2324, "mins_played")
print("Positions values (Belgium 23/24):", rows.Positions.astype(str).value_counts().head(15).to_dict())
rows[["ParticipantName", "TeamName", "Positions", "MinutesPlayed"]].head(5)

Positions values (Belgium 23/24):

{'[11]': 37, '[36]': 27, '[115]': 27, '[34]': 17, '[105]': 12, '[33]': 11, '[37]': 11, '[38]': 11, '[34, 32]': 10, '[106]': 9, '[35]': 9, '[32]': 8, '[3]': 7, '[64]': 6, '[104]': 6}

,ParticipantName,TeamName,Positions,MinutesPlayed
0,Maarten Vandevoordt,KRC Genk,[11],3690
1,Matte Smets,Sint-Truidense VV,[36],3600
2,Warleson,Cercle Brugge,[11],3600
3,Brandon Mechele,Club Brugge,[36],3583
4,Mathias Delorge,Sint-Truidense VV,"[64, 75]",3551


## Part 7 — ClubElo naming/coverage and the Transfermarkt injury page

ClubElo: one club history (columns, date intervals) and one date snapshot (naming convention,
countries covered, levels) — decides whether feeder clubs get Elo at all. Injury page: one
player's table, and what '?' and '-' mean in `until` / `Days` / `Games missed`; a player with a
long history to see whether the page is paginated.

In [67]:
from io import StringIO


def clubelo_get(key):
    time.sleep(3)
    response = requests.get(f"http://api.clubelo.com/{key}", headers=PLAIN_UA, timeout=90)
    return response.status_code, response.text


status, body = clubelo_get("Arsenal")
arsenal_elo = pd.read_csv(StringIO(body))
print(status, len(arsenal_elo), "rows |", arsenal_elo.From.min(), "→", arsenal_elo.To.max(), "| columns:", list(arsenal_elo.columns))
arsenal_elo.tail(5)

200

6507

rows |

1946-07-07

→

2026-12-31

| columns:

['Rank', 'Club', 'Country', 'Level', 'Elo', 'From', 'To']

,Rank,Club,Country,Level,Elo,From,To
6502,1.0,Arsenal,ENG,1,2065.758057,2026-05-25,2026-05-27
6503,1.0,Arsenal,ENG,1,2067.453613,2026-05-28,2026-05-30
6504,1.0,Arsenal,ENG,1,2063.758057,2026-05-31,2026-08-21
6505,1.0,Arsenal,ENG,1,2063.758057,2026-08-22,2026-08-30
6506,1.0,Arsenal,ENG,1,2063.758057,2026-08-31,2026-12-31


In [68]:
status, body = clubelo_get("2024-08-01")
elo_snapshot = pd.read_csv(StringIO(body))
print(status, len(elo_snapshot), "clubs on 2024-08-01 | levels:", elo_snapshot.Level.value_counts().to_dict())
print("clubs per country:", elo_snapshot.Country.value_counts().to_dict())

200

629

clubs on 2024-08-01 | levels:

{1: 421, 0: 106, 2: 102}

clubs per country:

{'ENG': 44, 'ESP': 42, 'ITA': 40, 'GER': 36, 'FRA': 36, 'TUR': 19, 'POL': 19, 'POR': 18, 'NED': 18, 'ROM': 17, 'BEL': 16, 'CZE': 16, 'RUS': 16, 'UKR': 16, 'NOR': 16, 'SRB': 16, 'BUL': 16, 'SWE': 16, 'GRE': 14, 'ISR': 14, 'DEN': 12, 'AUT': 12, 'SCO': 12, 'SUI': 12, 'HUN': 12, 'CRO': 10, 'SVN': 10, 'AZE': 4, 'CYP': 4, 'MOL': 4, 'SLK': 4, 'KAZ': 4, 'LAT': 4, 'FIN': 4, 'ARM': 4, 'BLR': 4, 'LIT': 4, 'BHZ': 4, 'KOS': 4, 'ISL': 4, 'FAR': 4, 'IRL': 4, 'ALB': 4, 'GEO': 4, 'EST': 4, 'MLT': 4, 'WAL': 4, 'NIR': 4, 'LUX': 4, 'MAC': 3, 'MNT': 3, 'GIB': 3, 'AND': 3, 'SMR': 3, 'LIE': 1}

In [69]:
ours = ["ENG", "ESP", "ITA", "GER", "FRA", "BEL", "NED", "POR", "TUR", "SUI", "AUT", "DEN", "BRA"]
print("countries missing from ClubElo:", [c for c in ours if c not in set(elo_snapshot.Country)])
elo_snapshot[elo_snapshot.Country.isin(ours) & (elo_snapshot.Level == 1)].groupby("Country").head(3).sort_values(["Country", "Rank"])[["Rank", "Club", "Country", "Level", "Elo"]]

countries missing from ClubElo:

['BRA']

,Rank,Club,Country,Level,Elo
76,77.0,Salzburg,AUT,1,1650.613647
119,NaN,Sturm Graz,AUT,1,1591.578491
180,NaN,LASK,AUT,1,1506.443237
39,40.0,Brugge,BEL,1,1727.673096
56,57.0,St Gillis,BEL,1,1694.669189
83,84.0,Anderlecht,BEL,1,1643.357056
75,76.0,FC Kobenhavn,DEN,1,1651.522095
96,97.0,Midtjylland,DEN,1,1619.640137
101,NaN,Nordsjaelland,DEN,1,1616.449341
0,1.0,Man City,ENG,1,2050.572998


In [70]:
kane_url = con.execute("SELECT url FROM players WHERE name = 'Harry Kane'").fetchone()[0].replace("/profil/", "/verletzungen/")
time.sleep(3)
response = requests.get(kane_url, headers=PLAIN_UA, timeout=30)
injury_tables = pd.read_html(StringIO(response.text))
kane_injuries = next(t for t in injury_tables if any("injury" in str(c).lower() for c in t.columns))
print(response.status_code, kane_url, "|", len(kane_injuries), "rows | columns:", list(kane_injuries.columns))
kane_injuries

200

https://www.transfermarkt.co.uk/harry-kane/verletzungen/spieler/132098

|

15

rows | columns:

['Season', 'Injury', 'from', 'until', 'Days', 'Games missed']

,Season,Injury,from,until,Days,Games missed
0,25/26,Ankle problems,31/03/2026,06/04/2026,7 days,2
1,25/26,Calf problems,05/03/2026,08/03/2026,4 days,1
2,24/25,Torn muscle fiber,01/12/2024,15/12/2024,15 days,4
3,23/24,Back problems,09/05/2024,01/06/2024,24 days,2
4,23/24,Ankle injury,17/03/2024,27/03/2024,11 days,2
5,20/21,Ankle injury,16/04/2021,22/04/2021,7 days,1
6,20/21,Ankle injury,29/01/2021,05/02/2021,8 days,2
7,20/21,unknown injury,30/11/2020,05/12/2020,6 days,1
8,19/20,Fitness,09/03/2020,01/04/2020,24 days,1
9,19/20,Torn thigh muscle,02/01/2020,09/03/2020,68 days,14


In [71]:
print("raw 'until':", kane_injuries["until"].astype(str).unique()[:10])
print("raw 'Days':", kane_injuries["Days"].astype(str).unique()[:10])
print("raw 'Games missed':", kane_injuries["Games missed"].astype(str).unique()[:10])

raw 'until':

<ArrowStringArray>
['06/04/2026', '08/03/2026', '15/12/2024', '01/06/2024', '27/03/2024', '22/04/2021', '05/02/2021', '05/12/2020', '01/04/2020', '09/03/2020']
Length: 10, dtype: str

raw 'Days':

<ArrowStringArray>
['7 days', '4 days', '15 days', '24 days', '11 days', '8 days', '6 days', '68 days', '52 days', '41 days']
Length: 10, dtype: str

raw 'Games missed':

<ArrowStringArray>
['2', '1', '4', '14', '9', '7', '-', '3']
Length: 8, dtype: str

In [72]:
# A long injury history: is the page paginated?
reus_url = con.execute("SELECT url FROM players WHERE name = 'Marco Reus'").fetchone()[0].replace("/profil/", "/verletzungen/")
time.sleep(3)
response = requests.get(reus_url, headers=PLAIN_UA, timeout=30)
reus_tables = pd.read_html(StringIO(response.text))
reus_injuries = next(t for t in reus_tables if any("injury" in str(c).lower() for c in t.columns))
print(response.status_code, len(reus_injuries), "rows | earliest season:", reus_injuries["Season"].astype(str).min(), "| pagination markers on page:", "page/2" in response.text or "pager" in response.text.lower())
reus_injuries.tail(3)

200

15

rows | earliest season:

21/22

| pagination markers on page:

True

,Season,Injury,from,until,Days,Games missed
12,21/22,Ill,07/03/2022,28/03/2022,22 days,3
13,21/22,minor knock,24/02/2022,07/03/2022,12 days,1
14,21/22,Knee problems,23/09/2021,27/09/2021,5 days,1


## Part 7b — Injury pagination and ClubElo key format

Criterion: if `/page/2` returns different rows, the injury pull walks pages until an empty or
repeated page; record how many pages a long history needs and whether `?` appears. ClubElo keys
for multi-word clubs are confirmed by a 200 with matching `Club` values; Brazil's absence by a
lookup of Flamengo.

In [73]:
def injury_page(base_url, page):
    time.sleep(3)
    url = base_url if page == 1 else f"{base_url}/page/{page}"
    response = requests.get(url, headers=PLAIN_UA, timeout=30)
    tables = pd.read_html(StringIO(response.text)) if response.status_code == 200 else []
    table = next((t for t in tables if any("injury" in str(c).lower() for c in t.columns)), pd.DataFrame())
    return response.status_code, table


for name, base_url in [("Harry Kane", kane_url), ("Marco Reus", reus_url)]:
    pages = []
    for page in range(1, 8):
        status, table = injury_page(base_url, page)
        if table.empty or (pages and table.equals(pages[-1])):
            break
        pages.append(table)
    history = pd.concat(pages, ignore_index=True) if pages else pd.DataFrame()
    print(name, "| pages fetched:", len(pages), "| rows:", len(history), "| seasons:", history["Season"].astype(str).min() if len(history) else None, "→", history["Season"].astype(str).max() if len(history) else None)
    print("   distinct 'Games missed' values:", sorted(history["Games missed"].astype(str).unique().tolist()) if len(history) else None)
    print("   distinct non-numeric 'Days':", [v for v in history["Days"].astype(str).unique() if "day" not in v] if len(history) else None)
    print("   distinct non-date 'until':", [v for v in history["until"].astype(str).unique() if "/" not in v] if len(history) else None)

Harry Kane

| pages fetched:

2

| rows:

21

...

['-', '1', '10', '2', '3', '30', '4', '40', '5', '7', '8', '9']

   distinct non-numeric 'Days':

[]

   distinct non-date 'until':

[]

In [74]:
for key in ["ManCity", "Man City", "ParisSG", "Paris SG", "FCKobenhavn", "Bueyueksehir", "Flamengo"]:
    status, body = clubelo_get(key)
    head = pd.read_csv(StringIO(body)) if status == 200 and body.startswith("Rank") else pd.DataFrame()
    print(f"{key:14s} {status} rows={len(head):5d}", "| club:", head.Club.iloc[0] if len(head) else body[:80].replace("\n", " "))

ManCity        200 rows= 6333

| club:

Man City

Man City       200 rows=    0

| club:

...

| club:

Bueyueksehir

Flamengo       200 rows=    0

| club:

Rank,Club,Country,Level,Elo,From,To   

**Note (Parts 7–7b, run locally 2026-08-27).** ClubElo: club history = intervals `From`/`To`
with `Elo`, `Rank` (top ~100 only), `Level`; Arsenal 6,507 intervals 1946 → 2026-12-31 (the
tail is a projection; reading Elo at past match dates never touches it). Snapshot 2024-08-01:
629 clubs, first divisions of all 12 European countries we need (BEL 16, NED 18, POR 18, TUR 19,
SUI/AUT/DEN 12); **Brazil absent** (`Flamengo` → 200 with an empty CSV). Key = display name
without spaces (`ManCity`, `ParisSG`, `FCKobenhavn`, `Bueyueksehir`); a wrong key also returns
200 + header only, so missing is detected from the empty body. Injury page: columns
`Season, Injury, from, until, Days, Games missed`; dates `dd/mm/yyyy`, `Days` always "N days",
`Games missed` numeric or `-`; **paginated at 15 rows** (`/page/N`): Kane 2 pages / 21 spells to
12/13, Reus 5 pages / 72 spells to 09/10 — Phase 0's "complete to 2009-10" held only for
players with < 15 spells. FotMob `Positions` is a list of numeric codes (`[34, 32]`), unmapped.

**DECIDED:** injury pull walks pages until empty/repeat; `Days` is the primary injury quantity,
`-` in `Games missed` → `NaN`; ClubElo key = name with spaces removed, empty CSV = missing,
Brazil → `NaN` opponent Elo (Tier-2 handling is Step 5a's decision). Step 1 complete: all six
sources read; fetch code (Step 2) is written against these notes.

# Step 2 — Fetch layer checks

Each fetcher lives in `src/scout/data/` with a unit test; this section imports it and records
what it returns on the real data, so the numbers the code is judged by are stored outputs
(CLAUDE.md §4: notebooks import from the package; §8: verify the port reproduces the numbers).

## 2.1 Transfermarkt — `scout.data.transfermarkt.load_player_club_seasons`

Expected: 33,061 appearance-based Big-5 rows (Phase 0), plus lineup-only rows with `NaN` minutes
that fill the appearance holes found in Part 3 (Atlético 2014-15 had 3 players).

In [75]:
from scout import config
from scout.data import transfermarkt as tm_loader

player_club_seasons = tm_loader.load_player_club_seasons(list(config.BIG5), config.SEASONS)
print("rows:", len(player_club_seasons), "| by source:", player_club_seasons.source.value_counts().to_dict(),
      "| players:", player_club_seasons.tm_player_id.nunique())
print("lineups-only rows with NaN minutes:", int(player_club_seasons[player_club_seasons.source == "lineups"].minutes.isna().sum()),
      "of", int((player_club_seasons.source == "lineups").sum()))
print("rows without a club name:", int(player_club_seasons.club_name.isna().sum()))
atletico_2014 = player_club_seasons[(player_club_seasons.club_id == 13) & (player_club_seasons.season == 2014)]
print("Atlético 2014-15 by source:", atletico_2014.source.value_counts().to_dict())

rows:

40299

| by source:

{'appearances': 33061, 'lineups': 7238}

| players:

...

7238

rows without a club name:

0

Atlético 2014-15 by source:

{'lineups': 23, 'appearances': 3}

## 2.2 Understat — `scout.data.understat.load` (fast kinds; per-match kinds land later)

Expected: 60 league-seasons per kind; 32,574 player-seasons (Phase 0); minutes per league-season
≈ 0.75M for 380-game leagues, 0.60M for the Bundesliga, 0.55M for the COVID-cut Ligue 1 2019-20.

In [76]:
from scout.data import understat

season_players = understat.load("player_season")
team_matches = understat.load("team_match")
schedules = understat.load("schedule")
print("player-seasons:", len(season_players), "| league-seasons:", season_players.groupby(["league", "season"]).ngroups,
      "| team-matches:", len(team_matches), "| scheduled matches:", len(schedules))
minutes = season_players.groupby(["league", "season"]).minutes.sum().unstack("league") / 1e6
minutes.round(2)

player-seasons:

32574

| league-seasons:

60

| team-matches:

21589

| scheduled matches:

21690

league,ENG-Premier League,ESP-La Liga,FRA-Ligue 1,GER-Bundesliga,ITA-Serie A
season,,,,,
2014,0.75,0.75,0.75,0.6,0.75
2015,0.75,0.75,0.75,0.61,0.75
2016,0.75,0.75,0.75,0.6,0.75
2017,0.75,0.75,0.75,0.6,0.75
2018,0.75,0.75,0.75,0.6,0.75
2019,0.75,0.75,0.55,0.6,0.75
2020,0.75,0.75,0.75,0.61,0.75
2021,0.75,0.75,0.75,0.61,0.75
2022,0.75,0.75,0.75,0.6,0.75


In [77]:
import glob

print({kind: len(glob.glob(f"data/raw/understat/{kind}/*.parquet")) for kind in understat.KINDS}, "← per-match kinds fill in as the background pull runs")

{'player_season': 0, 'player_match': 0, 'shots': 0, 'team_match': 0, 'schedule': 0}

← per-match kinds fill in as the background pull runs